# Pluto Propeller Dataset and Training — TVT folder edition
## 1. Scope and first run

This notebook reads your existing **TVT Split**, keeps train/valid/test assignments and the **ten references already selected**, obtains reviewed coordinates on matching clean originals, and optionally runs **A: bootstrap → B: train-only proposals → C: independent held-out labeling → D: final training/evaluation**. All expensive stages default off. Reports alone never mean training succeeded. No synthetic photos, seed reselection, automatic annotation-service uploads or drone-control commands are provided. Photograph only disarmed with motors stopped.

Classes: `0 propeller_cw`, `1 propeller_ccw`. Boxes enclose both blades and hub. Types need physical evidence under a confirmed viewing convention; never infer types/motor IDs from corners, generic motor maps or resting blade angle. Correct-only assemblies do not establish incorrect-installation sensitivity. A full verifier additionally needs official motor association, verified reference and sufficient front/completeness/orientation evidence. This development core remains **Unable to Verify** while those engines are missing.

Open at [Colab](https://colab.research.google.com/) → File → Upload notebook; save a Drive copy. Run sections in order. CPU works for preparation. Select Runtime → Change runtime type → GPU for explicitly enabled training. Helpers and exact repository v1 schemas are embedded; no clone is needed.

## 2. Install and inspect the runtime

Run once in standard Colab Python 3.10+. Existing torch/torchvision are constrained, not blindly upgraded. If resolution fails, use a compatible fresh runtime and keep the error. Restart if pip requests it, then rerun definitions. Package pins/API calls were checked, but Colab/GPU execution here is not claimed.

Official APIs: [YOLO11](https://docs.ultralytics.com/models/yolo11/), [train/resume](https://docs.ultralytics.com/modes/train/), [validation](https://docs.ultralytics.com/modes/val/), [prediction](https://docs.ultralytics.com/modes/predict/), [export](https://docs.ultralytics.com/modes/export/). Retain applicable software/weight licences for distribution.

In [ ]:
import sys, subprocess, tempfile, importlib.metadata as im
from pathlib import Path
if sys.version_info < (3, 10): raise RuntimeError("Use Colab Python 3.10+")
constraints = Path(tempfile.gettempdir()) / "pluto-torch-constraints.txt"
constraints.write_text("\n".join(f"{p}=={im.version(p)}" for p in ("torch", "torchvision")), encoding="utf-8")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints),
    "ultralytics==8.4.152", "Pillow==11.3.0", "PyYAML==6.0.3", "jsonschema==4.25.1", "nbformat==5.10.4"])


Run embedded definition cells as you reach them; expand to inspect code. Definitions never train or load models.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv
import hashlib
import io
import json
import math
import os
import platform
import random
import shutil
import stat
import subprocess
import sys
import time
import uuid
import zipfile

import numpy as np
from PIL import Image, ImageOps, ImageDraw
from jsonschema import Draft202012Validator

CLASSES = {0: "propeller_cw", 1: "propeller_ccw"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value, exclusive=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x" if exclusive else "w", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, allow_nan=False)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def read_csv(path):
    if not Path(path).is_file():
        return []
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def inside(root, relative):
    """Require a portable, relative path and defend against traversal/symlinks."""
    root = Path(root).resolve()
    raw = str(relative)
    posix = PurePosixPath(raw)
    if not raw or "\\" in raw or ":" in raw or posix.is_absolute() or ".." in posix.parts:
        raise ValueError(f"Unsafe relative path: {raw!r}")
    target = (root / raw).resolve()
    if not target.is_relative_to(root) or target == root:
        raise ValueError(f"Path escapes folder: {raw!r}")
    return target


def copy_once(source, target):
    """Reuse identical files only. A conflicting file is never overwritten."""
    source, target = Path(source), Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        if sha256(source) != sha256(target):
            raise ValueError(f"Conflicting existing file: {target}")
        return
    with source.open("rb") as src, target.open("xb") as dst:
        shutil.copyfileobj(src, dst)


def working_path(project, row):
    """TVT rows point at read-only sources or explicit normalized derivatives."""
    if row.get("image_path"):
        return Path(row["image_path"])
    return inside(project, row["working_relpath"])


In [ ]:
import importlib.metadata as im
ENVIRONMENT = {"python": sys.version, "platform": platform.platform()}
ENVIRONMENT["distributions"] = {d.metadata["Name"]: d.version for d in im.distributions() if d.metadata.get("Name")}
try:
    import torch
    ENVIRONMENT.update(cuda_available=torch.cuda.is_available(), cuda_version=torch.version.cuda,
        gpu_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except ImportError:
    ENVIRONMENT.update(cuda_available=False, cuda_version=None, gpu_name=None)
print({k: v for k, v in ENVIRONMENT.items() if k != "distributions"})
if not ENVIRONMENT["cuda_available"]: print("CPU preparation works; explicit training will be slower.")


## 3. Edit settings, then mount Drive

Defaults match your folder. Source paths are checked **after mounting** and never silently created. Outputs stay separate. Keep all RUN switches false on first run. Later enable one stage at a time, run its cell, then turn it off again. Saved run manifests resolve actual checkpoints automatically; overrides remain `None` until you select an existing run. ZIP intake/search/inspection cells are removed; ZIP import stays disabled.

In [ ]:
MOUNT_DRIVE = True
DATASET_SOURCE = Path("/content/drive/MyDrive/TVT Split")
TRAIN_SOURCE = DATASET_SOURCE / "train"
SEED_SOURCE = TRAIN_SOURCE / "labeled"
VALID_SOURCE = DATASET_SOURCE / "valid"
TEST_SOURCE = DATASET_SOURCE / "test"
PROJECT = Path("/content/drive/MyDrive/Pluto_Propeller_Dataset")
LOCAL_ROOT = Path("/content/pluto_processing")
SOURCE_METADATA = DATASET_SOURCE / "image_metadata.csv"  # optional real capture-session metadata
SEED, SPLIT_VERSION = 42, "tvt-v1"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_ZIP_IMPORT = False  # direct-folder notebook; no ZIP intake
RUN_COORDINATE_IMPORT = False
RUN_PROMOTION = False
RUN_BOOTSTRAP_TRAINING = False
RUN_ASSISTED_LABELING = False
RUN_DATASET_EXPORT = False
RUN_FINAL_TRAINING = False
RUN_RESUME = False
RUN_VALIDATION = False
RUN_FINAL_TEST = False
RUN_UPLOAD_PREDICTION = False
RUN_EXPORT = False
RUN_EXPORT_PARITY = False
RUN_HANDOFF_BUNDLE = False
ENABLE_SYNTHETIC_EXPERIMENTS = False  # unavailable; this workflow remains real-only
REVIEW_DIR = PROJECT / "reviews/manual_or_imported"
COORDINATE_IMPORT_MAP = PROJECT / "reviews/annotation_queue/coordinate_import.csv"
HUMAN_APPROVAL_PATH = PROJECT / "reviews/human_approval.json"
APPROVED_MANIFESTS = []  # otherwise read saved reviews/approved_selection.json
BOOTSTRAP_RUN_MANIFEST = None
FINAL_RUN_MANIFEST = None
RESUME_RUN_MANIFEST = None  # actual interrupted run manifest required
RESUME_STAGE = "final"     # bootstrap or final
FREEZE_FILE = PROJECT / "reviews/frozen_test_choices.json"
NOTEBOOK_COPY = PROJECT / "Pluto_Propeller_Dataset_and_Training.ipynb"
BOOTSTRAP_EPOCHS, FINAL_EPOCHS, PATIENCE, BATCH_SIZE = 20, 100, 20, 8
PROPOSAL_CONFIDENCE, PREDICT_CONFIDENCE, NMS_IOU = 0.25, 0.25, 0.6
BLUR_THRESHOLD, MIN_BRIGHTNESS, MAX_BRIGHTNESS, NEAR_DUPLICATE_DISTANCE = 50.0, 35.0, 220.0, 4
random.seed(SEED)
np.random.seed(SEED)
if "torch" in globals():
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir(): raise RuntimeError("Mount Drive before checking sources")


In [ ]:
SOURCES = {"train": TRAIN_SOURCE, "references": SEED_SOURCE, "val": VALID_SOURCE, "test": TEST_SOURCE}
for source in SOURCES.values():
    if PROJECT.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(PROJECT.resolve()):
        raise ValueError("Processed PROJECT must be separate from source folders")
for folder in ("metadata", "working", "reviews", "reference", "runs", "exports"):
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR = PROJECT / "reviews/reports" / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_PATH = PROJECT / "metadata/image_metadata.csv"
SEED_MAP_PATH = PROJECT / "reviews/seed_correspondence.csv"
DECISIONS_PATH = PROJECT / "reviews/duplicate_decisions.json"
REFERENCE_PATH = PROJECT / "reference/reference_configuration.json"
CLASS_CONFIRMATION_PATH = PROJECT / "reference/class_confirmation.json"
SELECTION_PATH = PROJECT / "reviews/approved_selection.json"
SPLIT_PATH = PROJECT / "metadata" / f"split_{SPLIT_VERSION}.json"
save_json(REPORT_DIR / "environment.json", ENVIRONMENT)
(REPORT_DIR / "environment-requirements.txt").write_text("\n".join(f"{k}=={v}" for k, v in sorted(ENVIRONMENT["distributions"].items())) + "\n", encoding="utf-8")
EXECUTION = {}
def run_action(name, enabled, action, next_action):
    if not enabled:
        EXECUTION[name] = {"state": "disabled", "next_action": next_action}
        print(f"{name}: OFF. {next_action}")
        return None
    try:
        value = action()
        EXECUTION[name] = {"state": "completed", "completed_at": utc_now()}
        return value
    except (ValueError, OSError) as exc:
        EXECUTION[name] = {"state": "blocked", "reason": str(exc)}
        print(f"{name}: BLOCKED — {exc}")
    except Exception as exc:
        EXECUTION[name] = {"state": "failed", "reason": str(exc)}
        print(f"{name}: FAILED — {type(exc).__name__}: {exc}. Inspect logs; no successful run is claimed.")
print("Sources:", {k: str(v) for k, v in SOURCES.items()}, "Outputs:", PROJECT)
if RUN_ZIP_IMPORT: print("ZIP intake unavailable here. Set RUN_ZIP_IMPORT=False and use direct source folders.")


## 4. Inventory actual files

Run to read images directly inside train/ (excluding labeled/), valid/ and test/, with labeled/ inventoried separately as reference material. Planning counts are 50/20/20 plus 10 references; actual counts/formats/unreadable files/TXT/JSON files are reported at runtime. No source file is renamed or overwritten. Do not infer 100 unique photographs from 100 image files.

Optional real session metadata uses `original_relpath` (e.g. `train/name.jpg`) or exact `source_path`, with actual `session_id`, `camera_view`, `parent_image_id`, quality status and notes. Unknown sessions remain blank; split names never become session IDs.

In [ ]:
TVT_FIELDS = ["image_id", "original_filename", "original_relpath", "source_path", "original_sha256",
    "source_split", "role", "image_path", "working_filename", "working_relpath", "image_sha256",
    "pixel_sha256", "width", "height", "format", "exif_orientation", "transformation", "session_id",
    "parent_image_id", "camera_view", "quality_status", "provenance", "notes", "error"]
SEED_MAP_FIELDS = ["reference_id", "reference_path", "clean_image_id", "status", "reviewer", "evidence"]
IMPORT_FIELDS = ["image_id", "label_source_path", "format", "coordinate_space", "image_sha256", "annotation_source"]


def inspect_tvt_sources(dataset_source, sources, metadata_path, source_metadata=None):
    """Read immediate raw splits, inventory references separately, never create sources."""
    dataset_source = Path(dataset_source).resolve()
    previous = {r["source_path"]: r for r in read_csv(metadata_path)}
    supplied = read_csv(source_metadata) if source_metadata else []
    rows, reports, annotations = [], [], []
    for name, source in sources.items():
        source = Path(source).resolve()
        if not source.is_dir():
            reports.append({"folder": name, "expected_path": str(source), "exists": False,
                            "message": "Missing: mount Drive, check spelling/case, or edit this source path. Source not created."})
            continue
        files = sorted(source.rglob("*")) if name == "references" else sorted(source.iterdir())
        image_count, formats, unreadable, ignored = 0, Counter(), [], []
        for path in files:
            if not path.is_file():
                continue
            suffix = path.suffix.lower()
            if suffix in {".txt", ".json", ".yaml", ".yml", ".csv"}:
                entry = {"folder": name, "path": str(path), "extension": suffix}
                if suffix == ".txt":
                    try:
                        boxes, errors = parse_yolo(path)
                        entry.update(detection_rows=len(boxes), geometry_errors=errors)
                    except (UnicodeError, OSError) as exc:
                        entry["geometry_errors"] = [str(exc)]
                elif suffix == ".json":
                    try:
                        data = load_json(path)
                        entry["json_keys"] = list(data)[:20] if isinstance(data, dict) else "not an object"
                    except Exception as exc:
                        entry["json_error"] = str(exc)
                annotations.append(entry)
                continue
            if suffix not in IMAGE_EXTENSIONS | {".heic", ".heif", ".dng", ".raw"}:
                ignored.append(path.name)
                continue
            image_count += 1
            relative = path.relative_to(dataset_source).as_posix() if path.is_relative_to(dataset_source) else str(path)
            row = {k: "" for k in TVT_FIELDS}
            old = previous.get(str(path), {})
            for key in ("session_id", "parent_image_id", "camera_view", "quality_status", "notes"):
                row[key] = old.get(key, "")
            for meta in supplied:
                if meta.get("source_path") == str(path) or meta.get("original_relpath") == relative:
                    for key in ("session_id", "parent_image_id", "camera_view", "quality_status", "notes"):
                        if meta.get(key):
                            row[key] = meta[key]
            file_hash = sha256(path)
            row.update(image_id=("ref_" if name == "references" else "img_") + digest([relative, file_hash])[:20],
                original_filename=path.name, original_relpath=relative, source_path=str(path),
                original_sha256=file_hash, source_split="reference" if name == "references" else name,
                role="reference_only" if name == "references" else "clean_candidate", provenance="real")
            row["quality_status"] = row["quality_status"] or "pending"
            row["camera_view"] = row["camera_view"] or "unknown"
            try:
                if suffix not in IMAGE_EXTENSIONS:
                    raise ValueError("Unsupported phone format: convert a separate copy with provenance; keep original unchanged")
                with Image.open(path) as image:
                    image.load()
                    allowed = {"JPEG": {".jpg", ".jpeg"}, "PNG": {".png"}, "WEBP": {".webp"},
                               "BMP": {".bmp"}, "TIFF": {".tif", ".tiff"}}
                    if suffix not in allowed.get(image.format, set()):
                        raise ValueError("Filename extension differs from decoded encoding")
                    row.update(width=image.width, height=image.height, format=image.format,
                               exif_orientation=int(image.getexif().get(274, 1)))
                    formats[image.format] += 1
            except Exception as exc:
                row["error"] = str(exc)
                unreadable.append({"path": str(path), "error": str(exc)})
            rows.append(row)
        reports.append({"folder": name, "expected_path": str(source), "exists": True,
                        "image_files": image_count, "formats": dict(formats), "unreadable": unreadable,
                        "annotation_files": sum(a["folder"] == name for a in annotations), "ignored_files": ignored})
    write_csv(metadata_path, rows, TVT_FIELDS)
    return rows, reports, annotations


def normalize_tvt(project, rows):
    """Use clean source pixels directly when possible; rotate/convert derived copies only."""
    for row in rows:
        if row["error"]:
            continue
        path = Path(row["source_path"])
        with Image.open(path) as raw:
            orientation = int(raw.getexif().get(274, 1))
            if orientation not in (1, 3, 6, 8):
                row["error"] = "Reflected/unknown EXIF: manual resolution required; pixels never mirrored"
                continue
            image = ImageOps.exif_transpose(raw).convert("RGB")
            row["pixel_sha256"] = digest([image.size, sha_bytes(np.asarray(image).tobytes())])
            row.update(width=image.width, height=image.height)
            if row["role"] == "reference_only":
                # Reference overlays are display-only, never clean training candidates.
                row.update(image_path=str(path), working_relpath=row["original_relpath"],
                           working_filename=path.name, image_sha256=sha256(path), transformation="display-only reference; EXIF applied for display")
                continue
            if orientation == 1 and raw.mode == "RGB" and raw.format in {"JPEG", "PNG", "WEBP"}:
                row.update(image_path=str(path), working_relpath=row["original_relpath"], working_filename=path.name,
                           image_sha256=sha256(path), transformation="none; use original upright RGB pixels read-only")
            else:
                relative = f"working/{row['image_id']}.png"
                target = inside(project, relative)
                if target.exists():
                    with Image.open(target) as existing:
                        if existing.getexif().get(274, 1) != 1 or not np.array_equal(np.asarray(existing.convert("RGB")), np.asarray(image)):
                            raise ValueError(f"Conflicting normalized working image: {target}")
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with target.open("xb") as out:
                        image.save(out, format="PNG")
                row.update(image_path=str(target), working_relpath=relative, working_filename=target.name,
                           image_sha256=sha256(target), transformation=f"EXIF {orientation} -> upright RGB PNG; no reflection")
    return rows


def sha_bytes(payload):
    import hashlib
    return hashlib.sha256(payload).hexdigest()


def tvt_quality(rows, blur_threshold=50, min_brightness=35, max_brightness=220):
    reports = []
    for row in rows:
        if row["error"] or not row["image_path"]:
            continue
        with Image.open(row["image_path"]) as raw:
            scores = quality_scores(ImageOps.exif_transpose(raw))
        reports.append({"image_id": row["image_id"], "original_sha256": row["original_sha256"],
            "image_sha256": row["image_sha256"], "pixel_sha256": row["pixel_sha256"], **scores,
            "flags": (["blur_candidate"] if scores["laplacian_variance"] < blur_threshold else [])
                     + (["exposure_candidate"] if not min_brightness <= scores["brightness"] <= max_brightness else [])})
    return reports


def reference_correspondence(rows, scores, distance=12):
    score_by_id = {s["image_id"]: s for s in scores}
    candidates = []
    for ref in [r for r in rows if r["role"] == "reference_only" and not r["error"]]:
        for clean in [r for r in rows if r["role"] != "reference_only" and not r["error"]]:
            a, b = score_by_id.get(ref["image_id"]), score_by_id.get(clean["image_id"])
            if not a or not b:
                continue
            exact = ref["original_sha256"] == clean["original_sha256"] or ref["pixel_sha256"] == clean["pixel_sha256"]
            hamming = (int(a["dhash"], 16) ^ int(b["dhash"], 16)).bit_count()
            name_hint = Path(clean["original_filename"]).stem.lower() in Path(ref["original_filename"]).stem.lower()
            if exact or hamming <= distance or name_hint:
                candidates.append({"reference_id": ref["image_id"], "clean_image_id": clean["image_id"],
                    "clean_path": clean["source_path"], "split": clean["source_split"], "exact_pixels_or_bytes": exact,
                    "dhash_distance": hamming, "filename_hint": name_hint, "status": "candidate_requires_human_confirmation"})
    return candidates


def selected_seed_ids(rows, seed_map):
    refs = {r["image_id"]: r for r in rows if r["role"] == "reference_only"}
    clean = {r["image_id"]: r for r in rows if r["role"] != "reference_only"}
    errors, selected, seen = [], [], set()
    if len(refs) != 10:
        errors.append(f"Existing selected reference folder has {len(refs)} image files; expected the user's existing 10. Check folder; do not auto-reselect.")
    for entry in seed_map:
        ref_id = entry.get("reference_id")
        if ref_id not in refs or ref_id in seen:
            errors.append(f"Unknown/duplicate reference mapping: {ref_id}")
            continue
        seen.add(ref_id)
        row = clean.get(entry.get("clean_image_id"))
        if (not row or entry.get("status") != "confirmed" or not entry.get("reviewer") or not entry.get("evidence")
                or entry.get("reference_path") != refs[ref_id]["source_path"]):
            errors.append(f"Clean original required/uncertain for {refs[ref_id]['source_path']}; confirm correspondence in seed_correspondence.csv")
        elif row["source_split"] != "train" or row["error"]:
            errors.append(f"Seed {refs[ref_id]['source_path']} must map to a usable clean train/ photograph, never valid/test")
        else:
            selected.append(row["image_id"])
    for ref_id in refs.keys() - seen:
        errors.append(f"Clean original correspondence required for {refs[ref_id]['source_path']}")
    if len(selected) != len(set(selected)):
        errors.append("Two marked references map to one source photograph; these are not ten independent seeds")
    return selected, errors


In [ ]:
ROWS, SOURCE_REPORTS, ANNOTATION_FILES = inspect_tvt_sources(DATASET_SOURCE, SOURCES, METADATA_PATH, SOURCE_METADATA)
from IPython.display import display
display(SOURCE_REPORTS)
display(ANNOTATION_FILES)
save_json(REPORT_DIR / "source_inventory.json", {"folders": SOURCE_REPORTS, "annotation_files": ANNOTATION_FILES})
for report in SOURCE_REPORTS:
    if not report["exists"]: print(report["expected_path"], "—", report["message"])


## 5. Upright pixels without changing originals

Run to use upright RGB JPEG/PNG/WebP source pixels directly read-only. Only required EXIF rotation/mode/format conversion creates PNG derivatives under PROJECT/working/. Reflected/unknown EXIF blocks that image; no mirroring. Annotate the exact `image_path` and hash in metadata. Coordinates from rotated/cropped/resized originals cannot be reused without the matching transformation and human review.

labeled/ remains reference-only. Never remove markings to manufacture a clean original. Human `clean_image_confirmed=true` is required before any training/evaluation export.

In [ ]:
def quality_scores(image):
    gray = np.asarray(image.convert("L"), dtype=np.float64)
    if min(gray.shape) < 3:
        raise ValueError("Image too small for quality screening")
    laplacian = (gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2]
                 + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1])
    small = np.asarray(image.convert("L").resize((9, 8)), dtype=np.int16)
    bits = (small[:, 1:] > small[:, :-1]).ravel()
    dhash = sum(int(v) << i for i, v in enumerate(bits))
    return {"laplacian_variance": float(laplacian.var()), "brightness": float(gray.mean()),
            "dark_fraction": float((gray < 10).mean()), "bright_fraction": float((gray > 245).mean()),
            "dhash": f"{dhash:016x}"}


def quality_inventory(project, rows, blur_threshold=50, min_brightness=35, max_brightness=220):
    reports = []
    for row in rows:
        if row["error"] or not row["working_relpath"]:
            continue
        with Image.open(working_path(project, row)) as image:
            scores = quality_scores(image)
        flags = []
        if scores["laplacian_variance"] < blur_threshold:
            flags.append("blur_candidate")
        if not min_brightness <= scores["brightness"] <= max_brightness:
            flags.append("exposure_candidate")
        reports.append({"image_id": row["image_id"], **scores, "flags": flags,
                        "original_sha256": row["original_sha256"], "image_sha256": row["image_sha256"]})
    return reports


def duplicate_candidates(reports, distance=4):
    pairs = []
    for i, a in enumerate(reports):
        for b in reports[i + 1:]:
            exact = a["original_sha256"] == b["original_sha256"] or a["image_sha256"] == b["image_sha256"]
            hamming = (int(a["dhash"], 16) ^ int(b["dhash"], 16)).bit_count()
            if exact or hamming <= distance:
                pairs.append({"a": a["image_id"], "b": b["image_id"], "exact": exact, "distance": hamming})
    return pairs


def contact_sheet(project, rows, output, limit=24):
    selected = [r for r in rows if r.get("working_relpath") and not r.get("error")][:limit]
    if not selected:
        return None
    sheet = Image.new("RGB", (4 * 300, math.ceil(len(selected) / 4) * 250), "white")
    draw = ImageDraw.Draw(sheet)
    for i, row in enumerate(selected):
        with Image.open(working_path(project, row)) as image:
            thumb = ImageOps.contain(image.convert("RGB"), (290, 200))
        x, y = (i % 4) * 300, (i // 4) * 250
        sheet.paste(thumb, (x, y))
        draw.text((x + 3, y + 202), row["original_filename"][:40], fill="black")
        draw.text((x + 3, y + 220), row["image_id"], fill="black")
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output)
    return Path(output)


In [ ]:
ROWS = normalize_tvt(PROJECT, ROWS)
write_csv(METADATA_PATH, ROWS, TVT_FIELDS)
INVENTORY = [r for r in ROWS if r["role"] != "reference_only"]
REFERENCES = [r for r in ROWS if r["role"] == "reference_only"]
print("Raw clean-source candidates:", len(INVENTORY), "Reference-only images:", len(REFERENCES))
print("Errors:", [{"path": r["source_path"], "error": r["error"]} for r in ROWS if r["error"]])
for name, group in (("train", [r for r in INVENTORY if r["source_split"] == "train"]), ("references", REFERENCES), ("val", [r for r in INVENTORY if r["source_split"] == "val"]), ("test", [r for r in INVENTORY if r["source_split"] == "test"])):
    sheet = contact_sheet(PROJECT, group, REPORT_DIR / f"contact_{name}.jpg")
    if sheet:
        print(name, "— inspect clean status and quality")
        display(Image.open(sheet))


## 6. Quality and duplicate/correspondence candidates

Run screening, then inspect images. Exact byte/pixel hashes identify some duplicates; visual hashes and filename hints suggest correspondence between marked copies and clean originals. Candidates need human confirmation; overlays never add independent photographs. Different hashes can still describe the same capture.

Blur/exposure scores are heuristics, not guarantees. Mark exclusions `quality_status=rejected` in output metadata; keep deliberate hard images as `challenge` outside typed training. Nothing deletes originals. Cross-split near candidates need `decision: distinct`, human `reviewer` and `evidence` in duplicate_decisions.json keyed by sorted `image_id|image_id`. Exact duplicates cannot be waived.

In [ ]:
QUALITY = tvt_quality(ROWS, BLUR_THRESHOLD, MIN_BRIGHTNESS, MAX_BRIGHTNESS)
DUPLICATES = duplicate_candidates(QUALITY, NEAR_DUPLICATE_DISTANCE)
for i, a in enumerate(QUALITY):
    for b in QUALITY[i+1:]:
        if a["pixel_sha256"] == b["pixel_sha256"]:
            DUPLICATES.append({"a": a["image_id"], "b": b["image_id"], "exact": True, "distance": 0})
CORRESPONDENCE_CANDIDATES = reference_correspondence(ROWS, QUALITY)
save_json(REPORT_DIR / "quality_screening.json", QUALITY)
save_json(REPORT_DIR / "duplicate_candidates.json", DUPLICATES)
save_json(REPORT_DIR / "reference_correspondence_candidates.json", CORRESPONDENCE_CANDIDATES)
if not DECISIONS_PATH.exists(): save_json(DECISIONS_PATH, {})
display(CORRESPONDENCE_CANDIDATES)
print("Distinct decoded clean-source pixel hashes:", len({r["pixel_sha256"] for r in INVENTORY if not r["error"]}), "; human clean status/near-duplicate independence still unverified.")


## 7. Physical class evidence and pending official reference

Run to create pending templates only if absent. Repository reference v1 uses **status: pending/verified**, not `verified=false`; pending has no motor map. Separately record actual CW/CCW physical evidence, agreed viewing convention and reviewer in class_confirmation.json before training.

Your convention **wider end = head/front; narrower end = tail/rear** is recorded as user-provided, pending official verification. It is not a proven visual-front detector or motor mapping. Only actual official evidence can establish a verified reference. No screen-corner class pattern is propagated.

In [ ]:
REFERENCE_SCHEMA = json.loads('{\n  "$schema": "https://json-schema.org/draft/2020-12/schema",\n  "title": "Reference v1",\n  "type": "object",\n  "additionalProperties": false,\n  "required": [\n    "schema_version",\n    "reference_version",\n    "status",\n    "drone_variant",\n    "front_convention",\n    "source_evidence",\n    "supported_views",\n    "motors"\n  ],\n  "properties": {\n    "schema_version": {\n      "const": "1.0.0"\n    },\n    "reference_version": {\n      "type": "string",\n      "minLength": 1\n    },\n    "status": {\n      "enum": [\n        "pending",\n        "verified"\n      ]\n    },\n    "drone_variant": {\n      "type": [\n        "string",\n        "null"\n      ]\n    },\n    "front_convention": {\n      "type": [\n        "string",\n        "null"\n      ]\n    },\n    "source_evidence": {\n      "type": "array",\n      "items": {\n        "type": "string",\n        "minLength": 1\n      }\n    },\n    "supported_views": {\n      "type": "array",\n      "items": {\n        "type": "string",\n        "minLength": 1\n      }\n    },\n    "motors": {\n      "type": "array",\n      "items": {\n        "type": "object",\n        "additionalProperties": false,\n        "required": [\n          "motor_id",\n          "expected_type",\n          "expected_orientation"\n        ],\n        "properties": {\n          "motor_id": {\n            "type": "string",\n            "minLength": 1\n          },\n          "expected_type": {\n            "type": "string",\n            "minLength": 1\n          },\n          "expected_orientation": {\n            "type": "string",\n            "minLength": 1\n          }\n        }\n      }\n    }\n  }\n}\n')
RESULT_SCHEMA = json.loads('{\n  "$schema": "https://json-schema.org/draft/2020-12/schema",\n  "title": "VerificationResult v1",\n  "type": "object",\n  "additionalProperties": false,\n  "required": [\n    "schema_version",\n    "input_id",\n    "status",\n    "evidence_source",\n    "reference_version",\n    "engine_version",\n    "all_required_checks_observable",\n    "reasons",\n    "elapsed_ms",\n    "motors"\n  ],\n  "properties": {\n    "schema_version": {\n      "const": "1.0.0"\n    },\n    "input_id": {\n      "type": "string",\n      "minLength": 1\n    },\n    "status": {\n      "enum": [\n        "pass",\n        "incorrect_installation",\n        "unable_to_verify"\n      ]\n    },\n    "evidence_source": {\n      "enum": [\n        "real",\n        "fixture"\n      ]\n    },\n    "reference_version": {\n      "type": [\n        "string",\n        "null"\n      ]\n    },\n    "engine_version": {\n      "type": [\n        "string",\n        "null"\n      ]\n    },\n    "all_required_checks_observable": {\n      "type": "boolean"\n    },\n    "reasons": {\n      "type": "array",\n      "items": {\n        "type": "string"\n      },\n      "uniqueItems": true\n    },\n    "elapsed_ms": {\n      "type": [\n        "number",\n        "null"\n      ],\n      "minimum": 0\n    },\n    "motors": {\n      "type": "array",\n      "items": {\n        "type": "object",\n        "additionalProperties": false,\n        "required": [\n          "motor_id",\n          "bbox",\n          "observed_type",\n          "expected_type",\n          "observed_orientation",\n          "expected_orientation",\n          "finding",\n          "correction_code"\n        ],\n        "properties": {\n          "motor_id": {\n            "type": "string",\n            "minLength": 1\n          },\n          "bbox": {\n            "oneOf": [\n              {\n                "type": "null"\n              },\n              {\n                "type": "array",\n                "items": {\n                  "type": "number",\n                  "minimum": 0,\n                  "maximum": 1\n                },\n                "minItems": 4,\n                "maxItems": 4\n              }\n            ]\n          },\n          "observed_type": {\n            "type": [\n              "string",\n              "null"\n            ]\n          },\n          "expected_type": {\n            "type": [\n              "string",\n              "null"\n            ]\n          },\n          "observed_orientation": {\n            "type": [\n              "string",\n              "null"\n            ]\n          },\n          "expected_orientation": {\n            "type": [\n              "string",\n              "null"\n            ]\n          },\n          "finding": {\n            "enum": [\n              "match",\n              "mismatch",\n              "unknown"\n            ]\n          },\n          "correction_code": {\n            "type": [\n              "string",\n              "null"\n            ]\n          }\n        }\n      }\n    }\n  }\n}\n')


In [ ]:
def validate_reference(reference, schema):
    Draft202012Validator(schema).validate(reference)
    if reference["status"] == "pending" and reference["motors"]:
        raise ValueError("Pending reference must have no motor mapping.")
    if reference["status"] == "verified":
        if not all(reference[k] for k in ("drone_variant", "front_convention", "source_evidence", "supported_views", "motors")):
            raise ValueError("Verified reference requires physical evidence and complete configuration.")
        ids = [m["motor_id"] for m in reference["motors"]]
        if len(ids) != len(set(ids)):
            raise ValueError("Duplicate reference motor IDs")
    return reference


def classes_confirmed(declaration):
    return (declaration.get("status") == "confirmed" and bool(declaration.get("reviewer"))
            and bool(declaration.get("viewing_convention")) and bool(declaration.get("source_evidence"))
            and declaration.get("classes") == {str(k): v for k, v in CLASSES.items()})


In [ ]:
PENDING_REFERENCE = dict(schema_version="1.0.0", reference_version="pending", status="pending", drone_variant=None, front_convention=None, source_evidence=[], supported_views=[], motors=[])
if not REFERENCE_PATH.exists(): save_json(REFERENCE_PATH, PENDING_REFERENCE)
if not CLASS_CONFIRMATION_PATH.exists():
    save_json(CLASS_CONFIRMATION_PATH, {"status": "pending", "classes": {str(k): v for k, v in CLASSES.items()}, "viewing_convention": "", "source_evidence": [], "reviewer": ""})
CONVENTION_PATH = PROJECT / "reference/user_body_convention.json"
if not CONVENTION_PATH.exists():
    save_json(CONVENTION_PATH, {"status": "pending_official_verification", "source": "user-provided", "head_front": "wider end", "tail_rear": "narrower end", "visual_front_detector_verified": False})
try: REFERENCE = validate_reference(load_json(REFERENCE_PATH), REFERENCE_SCHEMA)
except Exception as exc:
    REFERENCE = PENDING_REFERENCE
    print("Invalid reference; verification blocked:", exc)
CLASS_DECLARATION = load_json(CLASS_CONFIRMATION_PATH)
print("Reference:", REFERENCE["status"], "Class evidence confirmed:", classes_confirmed(CLASS_DECLARATION))


## 8. Keep the ten references; obtain coordinates on matching clean originals

Run templates. In reviews/seed_correspondence.csv, link **each already-selected reference** to its clean train/ image ID; record `status=confirmed`, human reviewer and evidence. Suggestions in section 6 are not automatic matches. Valid/test photos cannot be training seeds. Missing originals are reported by exact reference path; do not choose replacements or erase markings.

**If labeled/ contains images only:** drawn boxes/text are reference overlays, not YOLO coordinates. Stage A stays blocked. Use this practical annotation import route:

1. Locate the selected clean `image_path` and hash in annotation_image_manifest.csv. Download those exact upright images via Colab's Files pane or open their Drive copies locally.
2. In a free local YOLO detection editor such as LabelImg or CVAT, use classes.txt in its fixed order. Use overlays as guides while drawing/checking complete-propeller boxes on the clean originals. Unknown types stay unresolved. No automatic service upload is performed.
3. Save/export YOLO TXT to your Drive annotation folder. In annotation_queue/coordinate_import.csv, set the actual absolute `label_source_path`, matching image ID/hash, `coordinate_space=working`, `format=yolo_txt`, and annotation source manual/claude/model. Original TXT stems are allowed because pairing is explicit. Enable RUN_COORDINATE_IMPORT and run its cell.
4. Review/correct section 9 overlays and approve actual ground truth. Existing correct coordinate files can be imported without redoing annotation.

Discovery reports show TXT geometry/JSON keys, but cannot establish image/class/coordinate correspondence. Supported `normalized_boxes_json` explicitly declares matching `image_sha256`, `coordinate_space: working`, `classes: {"0":"propeller_cw","1":"propeller_ccw"}` and `boxes` entries with integer `class_id`, `x_center,y_center,width,height`. Other JSON/polygons need explicit documented conversion; nothing guesses coordinates from pixels or arrow labels.

In [ ]:
IMAGE_REVIEW_FIELDS = [
    "image_id", "original_relpath", "working_relpath", "image_sha256", "label_relpath",
    "label_sha256", "coordinate_space", "annotation_source", "is_seed", "review_status",
    "reviewer", "reviewed_at", "quality_status", "completeness_confirmed",
    "unresolved_visible_targets", "front_visibility", "physical_installation_status",
    "physical_status_evidence", "notes",
    "clean_image_confirmed", "independent_review", "independent_reviewer", "independent_review_evidence",
]
OBJECT_REVIEW_FIELDS = ["image_id", "row_index", "label_sha256", "class_confirmed", "type_evidence",
                        "motor_id", "occluded", "truncated", "reviewer", "notes"]


def parse_yolo(path):
    boxes, errors = [], []
    if not Path(path).is_file():
        return [], ["Missing label file; this is NOT a negative example."]
    lines = Path(path).read_text(encoding="utf-8-sig").splitlines()
    seen = set()
    for index, line in enumerate(lines):
        fields = line.split()
        if len(fields) != 5:
            errors.append(f"Line {index + 1}: expected 5 detection fields, polygons/blank rows rejected")
            continue
        try:
            if fields[0] not in {"0", "1"}:
                raise ValueError("class must be 0 or 1")
            values = tuple(float(value) for value in fields[1:])
            if not all(math.isfinite(v) and 0 <= v <= 1 for v in values):
                raise ValueError("non-finite/out-of-range coordinate")
            x, y, width, height = values
            if width <= 0 or height <= 0 or x - width / 2 < -1e-7 or y - height / 2 < -1e-7 or x + width / 2 > 1 + 1e-7 or y + height / 2 > 1 + 1e-7:
                raise ValueError("non-positive box or box outside image")
            box = (int(fields[0]), *values)
            if box in seen:
                raise ValueError("duplicate annotation row")
            seen.add(box)
            boxes.append(box)
        except ValueError as exc:
            errors.append(f"Line {index + 1}: {exc}")
    if not lines:
        errors.append("Empty labels are not supported for this drone-only typed seed workflow.")
    return boxes, errors


def annotation_batch(project, inventory, review_dir):
    """Validate raw proposals without trusting AI-written approval fields."""
    project, review_dir = Path(project), Path(review_dir)
    by_id = {r["image_id"]: r for r in inventory}
    reviews = read_csv(review_dir / "image_review.csv")
    objects = read_csv(review_dir / "object_review.csv")
    report, seen, used_labels = [], set(), set()
    for review in reviews:
        image_id = review.get("image_id", "")
        errors = []
        row = by_id.get(image_id)
        if image_id in seen:
            errors.append("Duplicate image review row")
        seen.add(image_id)
        if row is None or row.get("error") or not row.get("working_relpath") or row.get("role") == "reference_only":
            report.append({"image_id": image_id, "errors": ["Unknown/unusable image ID; link to inventory first"], "boxes": []})
            continue
        label_path = None
        try:
            label_path = inside(review_dir, review.get("label_relpath", ""))
            if label_path.suffix.lower() != ".txt":
                raise ValueError("YOLO label must be .txt")
            if str(label_path) in used_labels:
                raise ValueError("One label file cannot label two images")
            used_labels.add(str(label_path))
            boxes, label_errors = parse_yolo(label_path)
            errors.extend(label_errors)
        except (ValueError, OSError, UnicodeError) as exc:
            errors.append(str(exc))
            boxes = []
        image_path = working_path(project, row)
        actual_image_hash = sha256(image_path)
        actual_label_hash = sha256(label_path) if label_path and label_path.is_file() else None
        for field in ("original_relpath", "working_relpath"):
            if review.get(field) != row[field]:
                errors.append(f"{field} does not match inventory")
        if review.get("image_sha256") != actual_image_hash or row["image_sha256"] != actual_image_hash:
            errors.append("Working-image hash mismatch; review the exact upright image")
        if review.get("label_sha256") != actual_label_hash:
            errors.append("Label hash mismatch; invalidate stale reviews")
        if review.get("coordinate_space") != "working":
            errors.append("Labels must refer to exact upright working pixels; no coordinate reuse after transforms")
        if review.get("annotation_source") not in ("claude", "manual", "model"):
            errors.append("Record annotation_source: claude/manual/model")
        if review.get("review_status") != "approved" or not all(review.get(k) for k in ("reviewer", "reviewed_at")):
            errors.append("Human image review pending")
        if review.get("quality_status") != "approved":
            errors.append("Quality pending/rejected/challenge: excluded from typed training")
        if row.get("source_split"):
            if review.get("clean_image_confirmed") != "true":
                errors.append("Confirm clean original pixels; burned-in annotations cannot enter training/evaluation")
            if row["source_split"] in {"val", "test"} and (review.get("independent_review") != "true"
                    or not review.get("independent_reviewer") or not review.get("independent_review_evidence")):
                errors.append("Held-out labels require independent human correction/checking evidence, including any model-assisted labels")
        if review.get("completeness_confirmed") != "true" or review.get("unresolved_visible_targets") != "0":
            errors.append("Missing/unresolved visible targets: exclude until human resolution")
        object_rows = [o for o in objects if o.get("image_id") == image_id]
        indices = [o.get("row_index") for o in object_rows]
        if sorted(indices) != sorted(str(i) for i in range(len(boxes))):
            errors.append("One object-review row required per YOLO row; row_index is zero-based")
        for obj in object_rows:
            if obj.get("label_sha256") != actual_label_hash or obj.get("class_confirmed") != "true" or not obj.get("type_evidence") or not obj.get("reviewer"):
                errors.append("Object class/evidence/hash/human review missing")
            if obj.get("occluded") != "false" or obj.get("truncated") != "false":
                errors.append("Baseline excludes occluded/truncated targets; do not invent blade extents")
        report.append({"image_id": image_id, "image": row, "review": review,
                       "objects": object_rows, "label_path": str(label_path) if label_path else None,
                       "image_sha256": actual_image_hash, "label_sha256": actual_label_hash,
                       "boxes": boxes, "errors": errors})
    unknown_objects = sorted({o.get("image_id", "") for o in objects} - seen)
    if unknown_objects:
        report.append({"image_id": "orphan_object_rows", "errors": [f"Unknown object image IDs: {unknown_objects}"], "boxes": []})
    return report


def overlay_annotations(project, item, output):
    if "image" not in item:
        return None
    with Image.open(working_path(project, item["image"])) as raw:
        image = raw.convert("RGB")
    draw = ImageDraw.Draw(image)
    for i, (cls, x, y, width, height) in enumerate(item["boxes"]):
        rect = ((x - width / 2) * image.width, (y - height / 2) * image.height,
                (x + width / 2) * image.width, (y + height / 2) * image.height)
        draw.rectangle(rect, outline="lime" if cls == 0 else "cyan", width=3)
        draw.text((rect[0], rect[1]), f"{i}: {CLASSES[cls]}", fill="red")
    image.thumbnail((1400, 1400))
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    image.save(output)
    return Path(output)


def batch_digest(report, declaration):
    return digest({"class_declaration": declaration,
                   "records": [{k: r.get(k) for k in ("image_id", "image", "review", "objects", "image_sha256", "label_sha256", "errors")}
                               for r in report]})


def promote_reviewed(project, report, declaration, approval, approved_root):
    if not report or any(r["errors"] for r in report):
        raise ValueError("Resolve all annotation/review errors before promotion")
    if not classes_confirmed(declaration):
        raise ValueError("Physical CW/CCW class confirmation is pending")
    batch_hash = batch_digest(report, declaration)
    if (approval.get("batch_sha256") != batch_hash or approval.get("decision") != "APPROVE_REVIEWED_LABELS"
            or not approval.get("approver") or not approval.get("approved_at")):
        raise ValueError("Explicit human approval for this exact batch is required")
    root = Path(approved_root) / batch_hash
    if root.exists():
        raise ValueError("Approved batch already exists; retain it and select its manifest")
    root.mkdir(parents=True)
    records = []
    for item in report:
        # Recheck immediately before copying: prior validation is not a durable approval.
        if sha256(working_path(project, item["image"])) != item["image_sha256"] or sha256(item["label_path"]) != item["label_sha256"]:
            raise ValueError("Input changed after review; rerun validation")
        label = root / "labels" / f"{item['image_id']}.txt"
        copy_once(item["label_path"], label)
        records.append({**item["image"], "image_sha256": item["image_sha256"],
                        "label_path": str(label), "label_sha256": item["label_sha256"],
                        "review": item["review"], "objects": item["objects"]})
    manifest = root / "manifest.json"
    save_json(manifest, {"batch_sha256": batch_hash, "class_sha256": digest(declaration),
                         "approval": approval, "records": records}, exclusive=True)
    return manifest


def approved_records(project, manifests, declaration):
    records = {}
    if not classes_confirmed(declaration):
        return []
    for manifest in manifests:
        data = load_json(manifest)
        if data["class_sha256"] != digest(declaration):
            raise ValueError("Class confirmation changed; re-review labels")
        reconstructed = []
        for row in data["records"]:
            image = {k: v for k, v in row.items() if k not in {"review", "objects", "label_path", "label_sha256"}}
            reconstructed.append({"image_id": row["image_id"], "image": image,
                                  "review": row["review"], "objects": row["objects"],
                                  "image_sha256": row["image_sha256"], "label_sha256": row["label_sha256"], "errors": []})
        approval = data.get("approval", {})
        if (batch_digest(reconstructed, declaration) != data["batch_sha256"]
                or approval.get("batch_sha256") != data["batch_sha256"]
                or approval.get("decision") != "APPROVE_REVIEWED_LABELS"
                or not approval.get("approver") or not approval.get("approved_at")):
            raise ValueError("Approved manifest/review metadata changed; re-review required")
        for row in data["records"]:
            if row["image_id"] in records:
                raise ValueError("Multiple approved versions of an image: select one explicitly")
            if sha256(working_path(project, row)) != row["image_sha256"] or sha256(row["label_path"]) != row["label_sha256"]:
                raise ValueError("Approved image/label changed; approval invalid")
            records[row["image_id"]] = row
    return list(records.values())


In [ ]:
def review_template(row, is_seed=False, source="manual"):
    result = {k: "" for k in IMAGE_REVIEW_FIELDS}
    result.update({k: row.get(k, "") for k in ("image_id", "original_relpath", "working_relpath", "image_sha256")})
    result.update(label_relpath=f"labels/{row['image_id']}.txt", coordinate_space="working",
        annotation_source=source, is_seed="true" if is_seed else "false", review_status="pending",
        quality_status="pending", completeness_confirmed="false", unresolved_visible_targets="unknown",
        clean_image_confirmed="false", independent_review="false")
    return result


def annotation_import_queue(rows, seeds, destination):
    """Write templates only once; no empty label files pretending to be negatives."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    usable = [r for r in rows if r["role"] != "reference_only" and not r["error"]]
    if not (destination / "image_review.csv").exists():
        write_csv(destination / "image_review.csv", [review_template(r, r["image_id"] in seeds) for r in usable], IMAGE_REVIEW_FIELDS)
    if not (destination / "object_review.csv").exists():
        write_csv(destination / "object_review.csv", [], OBJECT_REVIEW_FIELDS)
    if not (destination / "coordinate_import.csv").exists():
        write_csv(destination / "coordinate_import.csv", [dict(image_id=r["image_id"], label_source_path="",
            format="yolo_txt", coordinate_space="working", image_sha256=r["image_sha256"], annotation_source="manual") for r in usable], IMPORT_FIELDS)
    write_csv(destination / "annotation_image_manifest.csv", usable, TVT_FIELDS)
    return destination


def import_coordinate_labels(rows, mapping_path, destination, seeds):
    """Explicit image/hash/class/space mapping; never derive boxes from burned-in overlays."""
    by_id = {r["image_id"]: r for r in rows if r["role"] != "reference_only"}
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    reviews, objects, imported = [], [], []
    for entry in read_csv(mapping_path):
        if not entry.get("label_source_path"):
            continue
        row = by_id.get(entry.get("image_id"))
        if not row or row["error"]:
            raise ValueError(f"Unknown/unusable clean image ID: {entry.get('image_id')}")
        if entry.get("coordinate_space") != "working" or entry.get("image_sha256") != sha256(working_path(None, row)):
            raise ValueError(f"Confirm exact upright pixel coordinates and image hash for {row['source_path']}")
        source = Path(entry["label_source_path"])
        label = destination / "labels" / (row["image_id"] + ".txt")
        label.parent.mkdir(parents=True, exist_ok=True)
        if entry["format"] == "yolo_txt":
            boxes, errors = parse_yolo(source)
            if errors:
                raise ValueError(f"{source}: {errors}")
            copy_once(source, label)
        elif entry["format"] == "normalized_boxes_json":
            data = load_json(source)
            if (data.get("image_sha256") != row["image_sha256"] or data.get("coordinate_space") != "working"
                    or data.get("classes") != {str(k): v for k, v in CLASSES.items()}):
                raise ValueError("JSON must explicitly declare matching working-image hash, coordinate space and class map")
            lines = []
            for box in data.get("boxes", []):
                cls = box["class_id"]
                if type(cls) is not int or cls not in CLASSES:
                    raise ValueError("JSON class_id must be integer 0/1")
                lines.append(f"{cls} " + " ".join(str(box[k]) for k in ("x_center", "y_center", "width", "height")))
            label.write_text("\n".join(lines) + "\n", encoding="utf-8")
            if parse_yolo(label)[1]:
                raise ValueError("Invalid normalized-box JSON geometry")
        else:
            raise ValueError("Unsupported annotation format; use detection YOLO TXT or documented normalized_boxes_json, never reinterpret polygons")
        review = review_template(row, row["image_id"] in seeds, entry.get("annotation_source") or "manual")
        review.update(label_sha256=sha256(label))
        reviews.append(review)
        boxes, _ = parse_yolo(label)
        for i in range(len(boxes)):
            objects.append(dict(image_id=row["image_id"], row_index=i, label_sha256=sha256(label),
                                class_confirmed="false", occluded="unknown", truncated="unknown"))
        imported.append({"image_id": row["image_id"], "source": str(source), "source_sha256": sha256(source)})
    if not reviews:
        raise ValueError("No coordinate files mapped. Images with drawn boxes/text are reference overlays, not coordinate labels")
    write_csv(destination / "image_review.csv", reviews, IMAGE_REVIEW_FIELDS)
    write_csv(destination / "object_review.csv", objects, OBJECT_REVIEW_FIELDS)
    save_json(destination / "import_manifest.json", {"status": "pending_human_review", "imported": imported})
    return destination


In [ ]:
if not SEED_MAP_PATH.exists():
    write_csv(SEED_MAP_PATH, [{"reference_id": r["image_id"], "reference_path": r["source_path"], "status": "pending"} for r in REFERENCES], SEED_MAP_FIELDS)
SEED_IDS, SEED_ERRORS = selected_seed_ids(ROWS, read_csv(SEED_MAP_PATH))
for error in SEED_ERRORS: print(error)
QUEUE = annotation_import_queue(ROWS, SEED_IDS, PROJECT / "reviews/annotation_queue")
(QUEUE / "classes.txt").write_text("propeller_cw\npropeller_ccw\n", encoding="utf-8")
print("Selected references:", len(REFERENCES), "Confirmed clean correspondences:", len(SEED_IDS), "Templates:", QUEUE)
if REFERENCES and not any(r["extension"] in {".txt", ".json"} for r in ANNOTATION_FILES):
    print("No coordinate files found. Reference images alone cannot train YOLO; follow section 8's clean-original annotation route.")
def do_coordinate_import():
    return import_coordinate_labels(ROWS, COORDINATE_IMPORT_MAP, PROJECT / "reviews/imports" / uuid.uuid4().hex, SEED_IDS)
imported = run_action("coordinate_import", RUN_COORDINATE_IMPORT, do_coordinate_import, "Map actual coordinate files, then enable RUN_COORDINATE_IMPORT.")
if imported:
    REVIEW_DIR = imported
    print("Pending import:", REVIEW_DIR, "Reuse this REVIEW_DIR after restart.")


## 9. Validate, visualize, then human approval

Run validation even while review is pending. It checks five fields, classes 0/1, finite normalized coordinates, positive in-bounds boxes, duplicate rows, pairing/hashes and per-object completeness evidence. Missing/empty labels never become background negatives. A human checks all visible propellers and physically confirmed type evidence.

Inspect each overlay, correct TXT, and refresh hashes from the validation report after edits. Image review requires reviewer/time, `review_status=approved`, `quality_status=approved`, `clean_image_confirmed=true`, `completeness_confirmed=true`, `unresolved_visible_targets=0`. Object review requires one zero-based row per TXT object, current label hash, class confirmation/evidence/reviewer, `occluded=false,truncated=false` for this baseline. Unknown/partial/occluded/truncated targets exclude the image; never invent hidden extents. Motor IDs are separate optional official evidence.

Held-out images additionally require `independent_review=true`, actual `independent_reviewer` and `independent_review_evidence`, including model-assisted labels. Predictions cannot approve themselves.

After zero errors, copy the real batch digest from approval_request.json into HUMAN_APPROVAL_PATH with `decision: APPROVE_REVIEWED_LABELS`, actual `approver` and `approved_at`. The notebook never creates approval. Enable RUN_PROMOTION only after visual review. Proposals remain separate; changed pixels/labels/review evidence invalidate approval. Selected snapshot paths are saved for restart; select one approved version per image if reapproving.

In [ ]:
ANNOTATION_REPORT = annotation_batch(PROJECT, INVENTORY, REVIEW_DIR)
save_json(REPORT_DIR / "annotation_validation.json", ANNOTATION_REPORT)
for item in ANNOTATION_REPORT:
    print(item["image_id"], "actual label hash:", item.get("label_sha256"), "errors:", item["errors"])
    overlay = overlay_annotations(PROJECT, item, REPORT_DIR / "annotation_overlays" / (item["image_id"] + ".jpg"))
    if overlay: display(Image.open(overlay))
save_json(REPORT_DIR / "approval_request.json", {"batch_sha256": batch_digest(ANNOTATION_REPORT, load_json(CLASS_CONFIRMATION_PATH)), "decision": "PENDING_HUMAN_REVIEW", "approver": "", "approved_at": ""})
if not ANNOTATION_REPORT: print("No coordinate/review rows at", REVIEW_DIR, "; import first. Nothing approved.")
def chosen_manifests():
    return APPROVED_MANIFESTS or (load_json(SELECTION_PATH) if SELECTION_PATH.is_file() else [])
def current_approved():
    return approved_records(PROJECT, chosen_manifests(), load_json(CLASS_CONFIRMATION_PATH))
def do_promote():
    manifest = promote_reviewed(PROJECT, annotation_batch(PROJECT, INVENTORY, REVIEW_DIR), load_json(CLASS_CONFIRMATION_PATH), load_json(HUMAN_APPROVAL_PATH), PROJECT / "reviews/approved")
    save_json(SELECTION_PATH, chosen_manifests() + [str(manifest)])
    print("Actual approved snapshot:", manifest)
    return manifest
run_action("promotion", RUN_PROMOTION, do_promote, "Complete human review/class evidence and exact approval, then enable RUN_PROMOTION.")
try: APPROVED = current_approved()
except (ValueError, OSError) as exc:
    APPROVED = []
    print("Approved selection blocked:", exc)
print("Approved pairs:", len(APPROVED))


## 10. Preserve TVT assignments and audit leakage

Run to save existing source/hash assignments: train→train, valid→val, test→test. There is no random redistribution or session-based replacement. Moved sources crossing a split require correction or a documented new evaluation version/path.

Missing session metadata means **session independence unverified**, not invented S01/train sessions. Supplied sessions or parent descendants spanning splits are flagged for explicit correction. Relevant exact/unresolved near-duplicate leakage blocks dependent stages. Review/exclude duplicates explicitly without silently reassigning the source split.

In [ ]:
def preserve_tvt_splits(rows, path, version="tvt-v1"):
    clean = [r for r in rows if r["role"] != "reference_only"]
    previous = load_json(path) if Path(path).is_file() else {"version": version, "files": []}
    if previous["version"] != version:
        raise ValueError("Split version changed; choose a separate manifest path and document the reason")
    history = defaultdict(set)
    for item in previous["files"]:
        history[item["original_sha256"]].add(item["source_split"])
    for row in clean:
        if row["original_sha256"] in history and row["source_split"] not in history[row["original_sha256"]]:
            raise ValueError(f"Source moved/duplicated into a different split: {row['source_path']}; preserve original assignments or document a new evaluation version")
    records = {r["image_id"]: r for r in previous["files"]}
    records.update({r["image_id"]: {k: r[k] for k in ("image_id", "original_relpath", "source_path", "original_sha256", "source_split")} for r in clean})
    manifest = {"version": version, "files": list(records.values()), "policy": "source folders preserved; valid maps to val; no session inference"}
    save_json(path, manifest)
    return manifest


def split_audit(rows):
    clean = [r for r in rows if r["role"] != "reference_only" and r["quality_status"] not in {"rejected", "challenge"}]
    sessions, errors = defaultdict(set), []
    for row in clean:
        if row["session_id"]:
            sessions[row["session_id"]].add(row["source_split"])
    for session, splits in sessions.items():
        if len(splits) > 1:
            errors.append(f"Supplied capture session {session!r} spans {sorted(splits)}; correct leakage explicitly, no automatic reassignment")
    by_id = {r["image_id"]: r for r in clean}
    for row in clean:
        parent = row["parent_image_id"]
        visited = {row["image_id"]}
        while parent:
            if parent not in by_id or parent in visited:
                errors.append(f"Missing/cyclic parent lineage for {row['source_path']}")
                break
            if by_id[parent]["source_split"] != row["source_split"]:
                errors.append(f"Derived image crosses its parent's split: {row['source_path']}")
            visited.add(parent)
            parent = by_id[parent]["parent_image_id"]
    return {"errors": errors, "session_independence": "unverified" if any(not r["session_id"] for r in clean) or not clean else "metadata supplied; physical independence still requires review",
            "sessions_missing": sum(not r["session_id"] for r in clean)}


def tvt_leakage(rows, candidates, decisions, selected_ids=None):
    by_id = {r["image_id"]: r for r in rows if r["role"] != "reference_only" and r["quality_status"] not in {"rejected", "challenge"}}
    errors = []
    for pair in candidates:
        a, b = by_id.get(pair["a"]), by_id.get(pair["b"])
        if not a or not b or a["source_split"] == b["source_split"]:
            continue
        if selected_ids is not None and not ({a["image_id"], b["image_id"]} & set(selected_ids)):
            continue
        key = "|".join(sorted((a["image_id"], b["image_id"])))
        decision = decisions.get(key, {})
        exact = pair["exact"] or a["pixel_sha256"] == b["pixel_sha256"]
        if exact or decision.get("decision") != "distinct" or not decision.get("reviewer") or not decision.get("evidence"):
            errors.append(f"Cross-split {'exact duplicate' if exact else 'near-duplicate candidate'}: {a['source_path']} <> {b['source_path']}; review/exclude explicitly")
    return errors


def prepare_tvt_dataset(project, records, rows, candidates, decisions, destination, required_splits=("train", "val"), seed_ids=None):
    current = {r["image_id"]: r for r in rows if r["role"] != "reference_only"}
    selected = [r for r in records if r["image_id"] in current and r["source_split"] in required_splits
                and current[r["image_id"]]["quality_status"] not in {"rejected", "challenge"}]
    if seed_ids is not None:
        if len(seed_ids) != 10 or len(set(seed_ids)) != 10:
            raise ValueError("Stage A needs the exact 10 human-confirmed reference-to-clean train correspondences")
        selected = [r for r in selected if r["image_id"] in seed_ids]
        if {r["image_id"] for r in selected} != set(seed_ids) or any(r["source_split"] != "train" for r in selected):
            missing = set(seed_ids) - {r["image_id"] for r in selected}
            raise ValueError(f"Stage A needs approved clean labels for all existing 10 training seeds; missing {sorted(missing)}")
    selected_ids = {r["image_id"] for r in selected}
    audit = split_audit(rows)
    if audit["errors"]:
        raise ValueError("; ".join(audit["errors"]))
    leakage = tvt_leakage(rows, candidates, decisions, selected_ids)
    if leakage:
        raise ValueError("; ".join(leakage))
    counts, image_counts = defaultdict(Counter), Counter()
    for row in selected:
        now = current[row["image_id"]]
        if any(now[k] != row[k] for k in ("source_split", "source_path", "session_id", "parent_image_id", "image_sha256")):
            raise ValueError("Approved source/split/session/geometry changed; re-review")
        if row["review"].get("clean_image_confirmed") != "true" or row["review"].get("completeness_confirmed") != "true" or row["review"].get("unresolved_visible_targets") != "0":
            raise ValueError("Only complete, human-confirmed clean images can be exported")
        if row["source_split"] in {"val", "test"} and (row["review"].get("independent_review") != "true"
                or not row["review"].get("independent_reviewer") or not row["review"].get("independent_review_evidence")):
            raise ValueError("Stage C: independently human-check held-out ground truth, even when proposals came from a model")
        if sha256(working_path(project, row)) != row["image_sha256"] or sha256(row["label_path"]) != row["label_sha256"]:
            raise ValueError("Approved pixels/labels changed")
        boxes, errors = parse_yolo(row["label_path"])
        if errors:
            raise ValueError(f"{row['image_id']}: {errors}")
        counts[row["source_split"]].update(b[0] for b in boxes)
        image_counts[row["source_split"]] += 1
    for split in required_splits:
        if not image_counts[split] or set(counts[split]) != {0, 1}:
            raise ValueError(f"Missing reviewed {split} image/label pairs with both confirmed classes (images={image_counts[split]}, classes={dict(counts[split])}); annotate/import then approve this split")
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    for split in ("train", "val", "test"):
        for folder in ("images", "labels"):
            (destination / folder / split).mkdir(parents=True)
    exported = []
    for row in selected:
        image = working_path(project, row)
        split, name = row["source_split"], row["image_id"]
        image_relative = f"images/{split}/{name}{image.suffix.lower()}"
        label_relative = f"labels/{split}/{name}.txt"
        copy_once(image, destination / image_relative)
        copy_once(row["label_path"], destination / label_relative)
        exported.append({**row, "split": split, "export_image": image_relative, "export_label": label_relative})
    import yaml
    config = {"path": str(destination.resolve()), "train": "images/train", "val": "images/val", "names": CLASSES}
    if seed_ids is not None:
        config["val"] = "images/train"  # loader bookkeeping only; no bootstrap validation claim
    if "test" in required_splits:
        config["test"] = "images/test"
    (destination / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    manifest = {"records": exported, "image_counts": dict(image_counts), "class_counts": {k: dict(v) for k, v in counts.items()},
        "purpose": "bootstrap_assistance" if seed_ids is not None else "reviewed_" + "_".join(required_splits),
        "session_independence": audit["session_independence"]}
    manifest["dataset_sha256"] = digest(manifest)
    save_json(destination / "dataset_manifest.json", manifest)
    return destination / "data.yaml", manifest


In [ ]:
try: SPLIT_LEDGER = preserve_tvt_splits(ROWS, SPLIT_PATH, SPLIT_VERSION)
except ValueError as exc:
    SPLIT_LEDGER = None
    print("Split ledger blocked:", exc)
SPLIT_AUDIT = split_audit(ROWS)
save_json(REPORT_DIR / "split_audit.json", {**SPLIT_AUDIT, "leakage": tvt_leakage(ROWS, DUPLICATES, load_json(DECISIONS_PATH))})
print("Source split counts:", dict(Counter(r["source_split"] for r in INVENTORY)))
print("Session audit:", SPLIT_AUDIT)
print("Duplicate warnings:", tvt_leakage(ROWS, DUPLICATES, load_json(DECISIONS_PATH)))


## 11. Stage A — bootstrap from the ten reviewed clean seeds

Enable RUN_BOOTSTRAP_TRAINING only after the existing ten references map to clean **training** originals, real class evidence is confirmed, and all seed coordinates are human-approved with both classes. Validation/test labels are not prerequisites. This is a small-data labeling experiment with no accuracy guarantee; test data is never used for tuning. Training-image diagnostics/bookkeeping are not independent validation.

Starts only from **yolo11n.pt**. Defaults: 640 pixels, batch 8, workers 0, fixed seeds, 20 bootstrap epochs. Flips, geometric/reflection, mosaic/mixup/cutmix/copy-paste augmentations are disabled; only modest saturation/brightness changes remain. Cloud loggers/automatic SDK dependency installation are disabled. Unique persistent Drive runs hold checkpoints/logs; data is staged locally.

In [ ]:
def yolo_api():
    """Use the pinned SDK locally, with automatic cloud loggers disabled."""
    import importlib.metadata as metadata
    if metadata.version("ultralytics") != "8.4.152":
        raise ValueError("Run dependency setup: this notebook targets ultralytics==8.4.152")
    os.environ["YOLO_AUTOINSTALL"] = "false"
    from ultralytics import YOLO, settings
    settings.update({k: False for k in ("sync", "clearml", "comet", "dvc", "mlflow", "raytune", "wandb", "tensorboard")})
    return YOLO


def require_custom_model(path):
    path = Path(path)
    if not path.is_file():
        raise ValueError("Custom checkpoint missing. Train explicitly or select a real best.pt; no pretrained inference fallback.")
    YOLO = yolo_api()
    model = YOLO(str(path), task="detect")
    if model.names != CLASSES:
        raise ValueError(f"Checkpoint classes are not the two agreed propeller classes: {model.names}")
    return model


def baseline_settings(seed=42):
    return dict(imgsz=640, epochs=100, patience=20, batch=8, workers=0,
                seed=seed, deterministic=True, pretrained=True, save=True, save_period=1,
                fliplr=0.0, flipud=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0,
                degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
                hsv_h=0.0, hsv_s=0.1, hsv_v=0.1, close_mosaic=0)




In [ ]:
def resolve_run(project, stage, explicit=None, require_complete=True):
    path = Path(explicit) if explicit else Path(project) / "runs" / f"latest_{stage}.json"
    if not path.is_file():
        raise ValueError(f"No saved {stage} run at {path}. Enable that training stage after its prerequisites, or set its actual run_manifest.json path")
    data = load_json(path)
    if "manifest_path" in data:
        path, data = Path(data["manifest_path"]), load_json(data["manifest_path"])
    if data.get("stage") != stage or (require_complete and data.get("state") != "completed"):
        raise ValueError(f"{stage} run is not completed: {path}; use explicit resume for interruptions")
    checkpoint = data.get("best_pt")
    if require_complete and (not checkpoint or not Path(checkpoint).is_file() or sha256(checkpoint) != data.get("best_sha256")):
        raise ValueError(f"Actual {stage} best.pt missing/changed; inspect {path}")
    return path, data


def train_tvt_stage(project, stage, yaml_path, dataset, settings, environment, resume_manifest=None):
    """Called only behind the notebook's explicit stage switches."""
    if resume_manifest:
        manifest_path, run = resolve_run(project, stage, resume_manifest, require_complete=False)
        if run["dataset_sha256"] != dataset["dataset_sha256"] or run["settings"] != settings or run["environment"] != environment:
            raise ValueError("Resume requires the recorded dataset, settings and compatible exact package/CUDA environment")
        if run["state"] == "completed" or not run.get("last_pt") or not Path(run["last_pt"]).is_file():
            raise ValueError("Select an interrupted run with an actual last.pt, not a completed run")
        recorded = Path(run["staging_path"])
        local_root = Path(yaml_path).parent.parent.resolve()
        if not recorded.resolve().is_relative_to(local_root):
            raise ValueError("Recorded staging path must be within the configured local processing root")
        if not recorded.exists():
            shutil.copytree(Path(yaml_path).parent, recorded)
        elif load_json(recorded / "dataset_manifest.json")["dataset_sha256"] != dataset["dataset_sha256"]:
            raise ValueError("Old staging folder conflicts with dataset")
        import yaml
        config = yaml.safe_load(Path(yaml_path).read_text())
        config["path"] = str(recorded)
        (recorded / "data.yaml").write_text(yaml.safe_dump(config), encoding="utf-8")
        model = require_custom_model(run["last_pt"])
        arguments = {"resume": True}
        output = manifest_path.parent
    else:
        output = Path(project) / "runs" / (stage + "_" + uuid.uuid4().hex)
        output.mkdir(parents=True, exist_ok=False)
        manifest_path = output / "run_manifest.json"
        run = {"stage": stage, "state": "started", "started_at": utc_now(), "pretrained": "yolo11n.pt",
               "dataset_sha256": dataset["dataset_sha256"], "dataset": dataset, "settings": settings,
               "environment": environment, "staging_path": str(Path(yaml_path).parent),
               "best_pt": None, "last_pt": None, "training_yaml_sha256": sha256(yaml_path)}
        save_json(manifest_path, run)
        shutil.copy2(yaml_path, output / "data.yaml")
        YOLO = yolo_api()
        model = YOLO("yolo11n.pt")
        arguments = dict(data=str(yaml_path), project=str(output), name="fit", exist_ok=False, **settings)
    save_json(Path(project) / "runs" / f"latest_incomplete_{stage}.json", {"manifest_path": str(manifest_path)})
    def capture_checkpoints(trainer):
        for key, filename in (("best_pt", "best.pt"), ("last_pt", "last.pt")):
            actual = Path(trainer.save_dir) / "weights" / filename
            if actual.is_file():
                run[key] = str(actual)
        save_json(manifest_path, run)
    model.add_callback("on_model_save", capture_checkpoints)
    try:
        model.train(**arguments)
        capture_checkpoints(model.trainer)
        if not run["best_pt"] or not run["last_pt"]:
            raise ValueError("Training returned without actual best.pt and last.pt; inspect logs")
        run.update(state="completed", completed_at=utc_now(), best_sha256=sha256(run["best_pt"]), last_sha256=sha256(run["last_pt"]))
        save_json(manifest_path, run)
        save_json(Path(project) / "runs" / f"latest_{stage}.json", {"manifest_path": str(manifest_path)})
    except Exception as exc:
        run.update(state="interrupted_or_failed", error=str(exc))
        save_json(manifest_path, run)
        raise
    return manifest_path, run


def propose_tvt_labels(project, rows, approved, model_path, destination, confidence=0.25, iou=0.6):
    model = require_custom_model(model_path)
    approved_ids = {r["image_id"] for r in approved}
    pool = [r for r in rows if r["role"] != "reference_only" and r["source_split"] == "train"
            and r["image_id"] not in approved_ids and not r["error"] and r["quality_status"] not in {"rejected", "challenge"}]
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    reviews, objects, scores = [], [], []
    for row in pool:
        prediction = model.predict(source=str(working_path(project, row)), conf=confidence, iou=iou,
                                   imgsz=640, rect=False, verbose=False)[0]
        label = destination / "labels" / (row["image_id"] + ".txt")
        label.parent.mkdir(exist_ok=True)
        lines = []
        for cls, box, score in zip(prediction.boxes.cls.cpu().tolist(), prediction.boxes.xywhn.cpu().tolist(), prediction.boxes.conf.cpu().tolist()):
            lines.append(f"{int(cls)} " + " ".join(f"{v:.8f}" for v in box))
            scores.append({"image_id": row["image_id"], "row_index": len(lines) - 1, "score": score})
        label.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        review = review_template(row, source="model")
        review["label_sha256"] = sha256(label)
        reviews.append(review)
        for i in range(len(lines)):
            objects.append(dict(image_id=row["image_id"], row_index=i, label_sha256=sha256(label), class_confirmed="false", occluded="unknown", truncated="unknown"))
        overlay_annotations(project, {"image": row, "boxes": parse_yolo(label)[0]}, destination / "overlays" / (row["image_id"] + ".jpg"))
    write_csv(destination / "image_review.csv", reviews, IMAGE_REVIEW_FIELDS)
    write_csv(destination / "object_review.csv", objects, OBJECT_REVIEW_FIELDS)
    save_json(destination / "proposal_manifest.json", {"created_at": utc_now(), "state": "pending_human_review",
        "model_sha256": sha256(model_path), "confidence": confidence, "nms_iou": iou, "image_count": len(pool), "scores": scores})
    return destination


In [ ]:
def refreshed():
    rows, _, _ = inspect_tvt_sources(DATASET_SOURCE, SOURCES, METADATA_PATH, SOURCE_METADATA)
    rows = normalize_tvt(PROJECT, rows)
    preserve_tvt_splits(rows, SPLIT_PATH, SPLIT_VERSION)
    scores = tvt_quality(rows, BLUR_THRESHOLD, MIN_BRIGHTNESS, MAX_BRIGHTNESS)
    pairs = duplicate_candidates(scores, NEAR_DUPLICATE_DISTANCE)
    for i, a in enumerate(scores):
        for b in scores[i+1:]:
            if a["pixel_sha256"] == b["pixel_sha256"]:
                pairs.append({"a": a["image_id"], "b": b["image_id"], "exact": True, "distance": 0})
    return rows, pairs
def stage_dataset(required=("train", "val"), bootstrap=False, destination=None):
    if not classes_confirmed(load_json(CLASS_CONFIRMATION_PATH)):
        raise ValueError(f"Confirm physical class evidence in {CLASS_CONFIRMATION_PATH}")
    rows, pairs = refreshed()
    seeds = None
    if bootstrap:
        seeds, errors = selected_seed_ids(rows, read_csv(SEED_MAP_PATH))
        if errors: raise ValueError("; ".join(errors))
    return prepare_tvt_dataset(PROJECT, current_approved(), rows, pairs, load_json(DECISIONS_PATH),
        destination or LOCAL_ROOT / ("dataset_" + uuid.uuid4().hex), required, seeds)
def settings_for(stage):
    settings = baseline_settings(SEED)
    settings.update(epochs=BOOTSTRAP_EPOCHS if stage == "bootstrap" else FINAL_EPOCHS,
        patience=0 if stage == "bootstrap" else PATIENCE, batch=BATCH_SIZE,
        device=0 if ENVIRONMENT["cuda_available"] else "cpu", val=stage != "bootstrap", plots=stage != "bootstrap")
    return settings
def train_selected(stage, resume=None):
    yaml_path, dataset = stage_dataset(("train",) if stage == "bootstrap" else ("train", "val"), stage == "bootstrap")
    manifest, run = train_tvt_stage(PROJECT, stage, yaml_path, dataset, settings_for(stage), ENVIRONMENT, resume)
    print("Training actually completed. Manifest:", manifest)
    print("Actual best.pt:", run["best_pt"], "Actual last.pt:", run["last_pt"])
    return manifest
if RUN_RESUME and (RUN_BOOTSTRAP_TRAINING or RUN_FINAL_TRAINING):
    raise ValueError("Enable either resume or new training, not both")
run_action("bootstrap_training", RUN_BOOTSTRAP_TRAINING, lambda: train_selected("bootstrap"),
    "Map/review the existing ten clean train seeds and class evidence, then enable RUN_BOOTSTRAP_TRAINING.")


## 12. Stage B — pending proposals for remaining training photos

Enable RUN_ASSISTED_LABELING after a completed bootstrap. Saved metadata resolves the actual checkpoint and verifies the exact seed dataset. Seeds/other approved images, all valid/test images, references, rejected/challenge and unreadable images are excluded. No self-training occurs. TXT, scores, provenance and separate overlays go to a new **pending** folder. Zero detections are review tasks, not accepted negatives.

Set REVIEW_DIR to the printed folder, correct boxes/types/completeness and repeat section 9 human approval. Poor predictions mean further manual correction. Preserve original reviewed seed labels.

In [ ]:
def do_proposals():
    manifest, run = resolve_run(PROJECT, "bootstrap", BOOTSTRAP_RUN_MANIFEST)
    _, seed_dataset = stage_dataset(("train",), bootstrap=True)
    if seed_dataset["dataset_sha256"] != run["dataset_sha256"]:
        raise ValueError("Checkpoint differs from current reviewed ten; select matching saved bootstrap or retrain explicitly")
    rows, _ = refreshed()
    output = propose_tvt_labels(PROJECT, rows, current_approved(), run["best_pt"], PROJECT / "reviews/training_proposals" / uuid.uuid4().hex, PROPOSAL_CONFIDENCE, NMS_IOU)
    provenance = load_json(output / "proposal_manifest.json")
    provenance.update(bootstrap_run_manifest=str(manifest), bootstrap_run_sha256=sha256(manifest), seed_dataset_sha256=seed_dataset["dataset_sha256"], environment=ENVIRONMENT)
    save_json(output / "proposal_manifest.json", provenance)
    print("Pending proposals:", output, "Set REVIEW_DIR here; correct and approve via section 9.")
    return output
run_action("assisted_labeling", RUN_ASSISTED_LABELING, do_proposals, "Complete Stage A, then enable RUN_ASSISTED_LABELING; manually review every proposal.")


## 13. Stage C — independently annotate held-out photos; export YOLO data

Use section 8's clean-image annotation/import route for valid/ and test/ with their original assignments. Independently correct missing/extra boxes and uncertain types, record human evidence and approve via section 9. Externally supplied model-assisted proposals may be imported but require independent human correction; the notebook's bootstrap proposals never touch these splits. Model predictions are not evaluation truth.

RUN_DATASET_EXPORT creates a new persistent `images/train,val,test`, `labels/train,val,test`, `data.yaml`, and a manifest linking collision-free IDs to original names/source paths. Only approved clean complete images with validated matching TXT enter exports; no unlabeled negatives. Both classes are required per requested split. Main training needs train+val, not test labels. Complete final testing additionally requires all eligible test photos reviewed, with exclusions explicit and reported.

In [ ]:
def do_dataset_export():
    yaml_path, dataset = stage_dataset(("train", "val", "test"), destination=PROJECT / "exports" / ("yolo_dataset_" + uuid.uuid4().hex))
    print("Actual YOLO dataset:", yaml_path.parent, "Images:", dataset["image_counts"], "Classes:", dataset["class_counts"])
    return yaml_path
run_action("dataset_export", RUN_DATASET_EXPORT, do_dataset_export, "Independently review held-out labels, then enable RUN_DATASET_EXPORT.")


## 14. Stage D — final training and restart/resume

Enable RUN_FINAL_TRAINING after reviewed clean train+validation labels and leakage checks. Missing test coordinates block **testing**, not this stage. Fine-tune YOLO11n with validation checkpoint selection; default 100 epochs/patience 20 are engineering settings. Skipped/blocked stages never report success.

After success, actual best.pt/last.pt/manifest paths are printed and saved through runs/latest_final.json (bootstrap similarly). Later stages resolve these records, with no placeholder weight paths or unrelated pretrained fallback. Overrides in section 3 select another actual run.

After Colab disconnect, remount and rerun preparation/definitions using saved approved selections. For interruption, select the actual manifest in runs/latest_incomplete_<stage>.json, set RESUME_RUN_MANIFEST/RESUME_STAGE and enable only RUN_RESUME. Matching dataset/settings/exact package/CUDA environment and actual last.pt are required. Local staging is recreated at its recorded path. The unfinished current epoch may be lost; completed runs are never overwritten or resumed here.

In [ ]:
run_action("final_training", RUN_FINAL_TRAINING, lambda: train_selected("final"),
    "Approve train and validation pairs, then enable RUN_FINAL_TRAINING; test labels are not required yet.")
def do_resume():
    if RESUME_STAGE not in {"bootstrap", "final"} or not RESUME_RUN_MANIFEST:
        raise ValueError("Set RESUME_STAGE and an actual interrupted RESUME_RUN_MANIFEST")
    return train_selected(RESUME_STAGE, RESUME_RUN_MANIFEST)
run_action("resume", RUN_RESUME, do_resume, "Restore the interrupted run's environment and recorded manifest, then enable RUN_RESUME.")


## 15. Validation, errors and frozen final test

Enable validation to compute actual precision/recall, mAP@0.5, mAP@0.5:0.95, per-class metrics, confusion matrices and ground-truth/prediction error overlays. Counts by split/class/view/session are saved. Tune thresholds only on validation. Real training curves display only when files exist.

Before RUN_FINAL_TEST, write FREEZE_FILE with the actual `model_sha256`, `dataset_sha256`, `confidence`, `nms_iou`, human `reviewer` and `frozen_at`. The test cell reports exact expected values when missing. All eligible test photos need independently reviewed truth (explicit rejected/challenge exclusions are reported). A receipt prevents repeated testing of the same model/dataset. Standard AP uses a low score floor; frozen confidence applies to displayed deployment examples. Never tune on test.

Detector metrics do not establish installation-verification performance. Correct-only assemblies leave real incorrect-installation sensitivity and false approvals unestablished. Missing outputs are never fabricated.

In [ ]:
def metric_report(metrics, model_path, dataset_manifest, split):
    evaluated = [int(v) for v in metrics.box.ap_class_index]
    per_class = {}
    for index, cls in enumerate(evaluated):
        p, r, ap50, ap = metrics.box.class_result(index)
        per_class[CLASSES[cls]] = dict(precision=float(p), recall=float(r), map50=float(ap50), map50_95=float(ap))
    return {"created_at": utc_now(), "split": split, "model_sha256": sha256(model_path),
            "dataset_sha256": dataset_manifest["dataset_sha256"],
            "precision": float(metrics.box.mp), "recall": float(metrics.box.mr),
            "map50": float(metrics.box.map50), "map50_95": float(metrics.box.map), "per_class": per_class,
            "scope": "detection only; incorrect-installation sensitivity and false approvals not established"}


def iou_xywh(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    intersection = max(0, min(ax + aw, bx + bw) - max(ax, bx)) * max(0, min(ay + ah, by + bh) - max(ay, by))
    union = aw * ah + bw * bh - intersection
    return intersection / union if union else 0.0


def inspect_errors(model, project, rows, output, conf=0.25, iou=0.6, limit=12):
    """Greedy IoU review aid, not a replacement for official AP metrics."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    summaries = []
    for row in rows[:limit]:
        result = model.predict(str(working_path(project, row)), conf=conf, iou=iou,
                               imgsz=640, rect=False, verbose=False)[0]
        truth, errors = parse_yolo(row["label_path"])
        if errors:
            raise ValueError(str(errors))
        def left(box):
            cls, x, y, w, h = box
            return cls, [x - w / 2, y - h / 2, w, h]
        gt = [left(box) for box in truth]
        predictions = [(int(c), [x - w / 2, y - h / 2, w, h]) for c, (x, y, w, h)
                       in zip(result.boxes.cls.cpu().tolist(), result.boxes.xywhn.cpu().tolist())]
        used, tags = set(), []
        for cls, box in predictions:
            matches = sorted(((iou_xywh(box, target), j) for j, (_, target) in enumerate(gt)), reverse=True)
            if not matches or matches[0][0] < 0.5:
                tags.append("unmatched_prediction")
            else:
                _, j = matches[0]
                if j in used:
                    tags.append("duplicate_detection")
                elif cls != gt[j][0]:
                    tags.append("incorrect_type")
                used.add(j)
        if len(used) < len(gt):
            tags.append("missed_detection")
        result.save(filename=str(output / f"{row['image_id']}_prediction.jpg"))
        overlay_annotations(project, {"image": row, "boxes": truth}, output / f"{row['image_id']}_ground_truth.jpg")
        summaries.append({"image_id": row["image_id"], "tags": tags, "review": "human_required"})
    save_json(output / "error_candidates.json", summaries)
    return summaries


In [ ]:
def evaluate_final(split):
    manifest, run = resolve_run(PROJECT, "final", FINAL_RUN_MANIFEST)
    yaml_path, dataset = stage_dataset(("train", "val") if split == "val" else ("train", "val", "test"))
    rows, _ = refreshed()
    if split == "test":
        eligible = {r["image_id"] for r in rows if r["source_split"] == "test" and r["quality_status"] not in {"rejected", "challenge"}}
        reviewed = {r["image_id"] for r in dataset["records"] if r["split"] == "test"}
        if eligible != reviewed: raise ValueError(f"Missing independently reviewed test pairs: {sorted(eligible-reviewed)}; annotate/review or explicitly document exclusions")
        expected = dict(model_sha256=run["best_sha256"], dataset_sha256=dataset["dataset_sha256"], confidence=PREDICT_CONFIDENCE, nms_iou=NMS_IOU)
        freeze = load_json(FREEZE_FILE) if FREEZE_FILE.is_file() else {}
        if any(freeze.get(k) != v for k, v in expected.items()) or not freeze.get("reviewer") or not freeze.get("frozen_at"):
            raise ValueError(f"Human freeze required at {FREEZE_FILE}: {expected}, plus reviewer/frozen_at")
        receipt = PROJECT / "reviews/test_receipts" / (digest({k: expected[k] for k in ("model_sha256", "dataset_sha256")}) + ".json")
        save_json(receipt, {**freeze, "state": "started", "started_at": utc_now()}, exclusive=True)
    output = PROJECT / "runs" / ("evaluation_" + split + "_" + uuid.uuid4().hex)
    model = require_custom_model(run["best_pt"])
    metrics = model.val(data=str(yaml_path), split=split, imgsz=640, batch=BATCH_SIZE, workers=0, conf=0.001, iou=NMS_IOU,
                        plots=True, project=str(output), name="metrics", exist_ok=False)
    report = metric_report(metrics, run["best_pt"], dataset, split)
    report.update(image_counts=dataset["image_counts"], class_counts=dataset["class_counts"], session_independence=dataset["session_independence"], run_manifest=str(manifest))
    save_json(output / "metric_report.json", report)
    selected = [r for r in dataset["records"] if r["split"] == split]
    save_json(output / "view_session_counts.json", [{"view": v, "session_id": s, "images": n} for (v, s), n in Counter((r["camera_view"], r["session_id"]) for r in selected).items()])
    inspect_errors(model, PROJECT, selected, output / "error_review", PREDICT_CONFIDENCE, NMS_IOU)
    for plot in sorted((output / "metrics").glob("*.png")): display(Image.open(plot))
    save_json(PROJECT / "runs" / f"latest_evaluation_{split}.json", {"report_path": str(output / "metric_report.json")})
    if split == "test": save_json(receipt, {**freeze, "state": "completed", "report_path": str(output / "metric_report.json")})
    print("Actual evaluation:", output, report)
    return output
run_action("validation", RUN_VALIDATION, lambda: evaluate_final("val"), "Complete final training/reviewed validation pairs, then enable RUN_VALIDATION.")
run_action("final_test", RUN_FINAL_TEST, lambda: evaluate_final("test"), "Independently review all eligible test pairs and freeze choices after validation, then enable RUN_FINAL_TEST.")
try:
    _, run = resolve_run(PROJECT, "final", FINAL_RUN_MANIFEST)
    plot = Path(run["best_pt"]).parent.parent / "results.png"
    if plot.is_file(): display(Image.open(plot))
except (ValueError, OSError): print("No actual completed final-run curves available.")


## 16. Uploaded images and shared future live-frame processing

process_frame accepts upright unmirrored RGB uint8 frames, returns exact v1 result fields with separate quality/detection diagnostics, and stays **Unable to Verify** while front/motor association/completeness/orientation engines are missing. Four detections or a close-up never prove whole-drone evidence; expected values never substitute observations.

Optional uploads resolve the actual final checkpoint when available; absent resources return honest missing-resource results. The same core can accept future Android camera frames. Hosted Colab cannot access your laptop camera with cv2.VideoCapture(0); no camera/FPS/Android claim is made. No photo/checkbox proves physical disarm.

In [ ]:
def process_frame(frame_rgb, input_id, reference, reference_schema, result_schema,
                  model=None, engine_version=None, conf=0.25, iou=0.6):
    """Development core: honest detections; motor/front/orientation engine still pending.

    RGB upright unmirrored pixels in; contract v1 result + separate diagnostics out.
    This detector alone cannot verify required installation evidence, so never Pass.
    """
    started = time.perf_counter()
    result = dict(schema_version="1.0.0", input_id=input_id, status="unable_to_verify",
                  evidence_source="real", reference_version=None, engine_version=engine_version,
                  all_required_checks_observable=False, reasons=[], elapsed_ms=None, motors=[])
    diagnostic = {"detections": [], "quality": None, "motor_assignment": "unavailable",
                  "orientation_evidence": "unknown"}
    try:
        validate_reference(reference, reference_schema)
        if reference["status"] == "verified":
            result["reference_version"] = reference["reference_version"]
        else:
            result["reasons"].append("REFERENCE_NOT_CONFIGURED")
    except Exception:
        result["reasons"].append("REFERENCE_NOT_CONFIGURED")
    try:
        array = np.asarray(frame_rgb)
        if array.ndim != 3 or array.shape[2] != 3 or array.dtype != np.uint8 or min(array.shape[:2]) < 3:
            raise ValueError("RGB uint8 image required")
        image = Image.fromarray(array)
        diagnostic["quality"] = quality_scores(image)
    except Exception:
        image = None
        result["reasons"].append("INVALID_INPUT")
    if model is None or not engine_version:
        result["reasons"].append("ENGINE_UNAVAILABLE")
    elif image is not None:
        try:
            if model.names != CLASSES:
                raise ValueError("Wrong detector classes")
            prediction = model.predict(image, conf=conf, iou=iou, imgsz=640, rect=False, verbose=False)[0]
            for cls, xyxy, score in zip(prediction.boxes.cls.cpu().tolist(), prediction.boxes.xyxyn.cpu().tolist(), prediction.boxes.conf.cpu().tolist()):
                x1, y1, x2, y2 = xyxy
                diagnostic["detections"].append({"class": CLASSES[int(cls)], "score": float(score),
                                                 "bbox": [x1, y1, x2 - x1, y2 - y1]})
        except Exception:
            result["reasons"].append("ENGINE_ERROR")
    # No corner sorting or count-based completeness; no expected-type substitution.
    result["reasons"].extend(["UNKNOWN_ORIENTATION", "INCOMPLETE_DRONE"])
    result["reasons"] = list(dict.fromkeys(result["reasons"]))
    result["elapsed_ms"] = (time.perf_counter() - started) * 1000
    Draft202012Validator(result_schema).validate(result)
    return {"result": result, "diagnostics": diagnostic}


In [ ]:
def do_upload():
    from google.colab import files
    detector, version = None, None
    try:
        _, run = resolve_run(PROJECT, "final", FINAL_RUN_MANIFEST)
        detector, version = require_custom_model(run["best_pt"]), run["best_sha256"]
    except (ValueError, OSError) as exc: print("Missing detector; abstention path:", exc)
    for name, payload in files.upload().items():
        with Image.open(io.BytesIO(payload)) as raw:
            if raw.getexif().get(274, 1) not in (1, 3, 6, 8): raise ValueError("Reflected/unknown EXIF: resolve manually")
            image = ImageOps.exif_transpose(raw).convert("RGB")
        reply = process_frame(np.asarray(image), "upload_" + uuid.uuid4().hex, REFERENCE, REFERENCE_SCHEMA, RESULT_SCHEMA, detector, version, PREDICT_CONFIDENCE, NMS_IOU)
        print(name, json.dumps(reply, indent=2))
        draw = ImageDraw.Draw(image)
        for detection in reply["diagnostics"]["detections"]:
            x, y, w, h = detection["bbox"]
            draw.rectangle((x*image.width, y*image.height, (x+w)*image.width, (y+h)*image.height), outline="cyan", width=3)
        display(image)
run_action("upload_prediction", RUN_UPLOAD_PREDICTION, do_upload, "Enable RUN_UPLOAD_PREDICTION for Colab's file dialog; uploads are not added to training.")


## 17. Optional ONNX export, parity and local handoff

Export resolves the saved completed final run, preserves best.pt and writes a fresh static FP32 1×3×640×640 ONNX (opset 17, external NMS, no simplification). Actual I/O shapes are inspected. Typical two-class YOLO11 output is [1,6,8400] (xywh+two scores, no objectness); inspect the actual tensor.

Preprocess upright RGB → aspect-preserving bilinear letterbox640/pad114 → NCHW float32 /255. Direct runtimes record exact scale/integer padding, convert xywh→xyxy, subtract pad, divide by scale, clamp/normalize to original analyzed dimensions, then class-aware NMS. Ultralytics predict(rect=False) handles this mapping. Convert OpenCV BGR to RGB before process_frame.

Parity matches actual ONNX/PyTorch validation predictions by class/IoU, independently of ordering. Provisional tolerances are not Android proof; empty predictions do not establish parity. App owner coordinates actual mobile runtime/licences/device tests.

Handoff is a local file allowlist: executable notebook, source/seed/review/split metadata, environment and actual checkpoints/export/metrics where present. Raw/reference photos and overlays are omitted. Save NOTEBOOK_COPY first. Nothing is published or uploaded automatically.

In [ ]:
EXPORTED_ONNX = None
def do_export():
    _, run = resolve_run(PROJECT, "final", FINAL_RUN_MANIFEST)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx==1.19.1", "onnxruntime==1.23.2"])
    import onnxruntime as ort
    output = PROJECT / "exports" / ("onnx_" + uuid.uuid4().hex)
    output.mkdir(parents=True)
    copy_once(run["best_pt"], output / "best.pt")
    exported = Path(require_custom_model(output / "best.pt").export(format="onnx", imgsz=640, batch=1, dynamic=False, simplify=False, nms=False, opset=17, half=False, device="cpu"))
    session = ort.InferenceSession(str(exported), providers=["CPUExecutionProvider"])
    actual_io = {"inputs": [{"name": t.name, "shape": t.shape, "type": t.type} for t in session.get_inputs()], "outputs": [{"name": t.name, "shape": t.shape, "type": t.type} for t in session.get_outputs()]}
    save_json(output / "export_manifest.json", {"checkpoint_sha256": run["best_sha256"], "onnx_sha256": sha256(exported), "classes": CLASSES, "io": actual_io,
        "preprocessing": "RGB letterbox640 pad114 NCHW FP32 /255 rect=False", "nms_embedded": False, "confidence": PREDICT_CONFIDENCE, "nms_iou": NMS_IOU,
        "environment": ENVIRONMENT, "onnxruntime_version": ort.__version__, "android_verified": False})
    save_json(PROJECT / "exports/latest_onnx.json", {"path": str(exported)})
    print("Actual ONNX:", exported, actual_io)
    return exported
EXPORTED_ONNX = run_action("onnx_export", RUN_EXPORT, do_export, "Complete final training, then enable RUN_EXPORT; export does not establish Android compatibility.")
def do_parity():
    exported = EXPORTED_ONNX or Path(load_json(PROJECT / "exports/latest_onnx.json")["path"])
    _, run = resolve_run(PROJECT, "final", FINAL_RUN_MANIFEST)
    provenance = load_json(exported.parent / "export_manifest.json")
    if provenance["checkpoint_sha256"] != run["best_sha256"] or provenance["onnx_sha256"] != sha256(exported): raise ValueError("Export/checkpoint hashes differ")
    _, dataset = stage_dataset(("train", "val"))
    pt, onnx = require_custom_model(run["best_pt"]), yolo_api()(str(exported), task="detect")
    reports = []
    def tl(b):
        x, y, w, h = b
        return [x-w/2, y-h/2, w, h]
    for row in [r for r in dataset["records"] if r["split"] == "val"][:10]:
        args = dict(source=str(working_path(PROJECT, row)), conf=PREDICT_CONFIDENCE, iou=NMS_IOU, imgsz=640, rect=False, device="cpu", verbose=False)
        a, b = pt.predict(**args)[0].boxes, onnx.predict(**args)[0].boxes
        ac, bc, ab, bb = a.cls.cpu().tolist(), b.cls.cpu().tolist(), a.xywhn.cpu().tolist(), b.xywhn.cpu().tolist()
        asc, bsc, remaining, matches = a.conf.cpu().tolist(), b.conf.cpu().tolist(), set(range(len(bc))), []
        for cls, box, score in zip(ac, ab, asc):
            options = [(iou_xywh(tl(box), tl(bb[j])), j) for j in remaining if bc[j] == cls]
            if options:
                overlap, j = max(options)
                remaining.remove(j)
                matches.append({"iou": overlap, "score_difference": abs(score-bsc[j])})
        passed = bool(ac) and len(ac) == len(bc) == len(matches) and all(m["iou"] >= 0.99 and m["score_difference"] <= 0.01 for m in matches)
        reports.append({"image_id": row["image_id"], "pytorch_count": len(ac), "onnx_count": len(bc), "matches": matches, "provisional_tolerance_pass": passed})
    save_json(exported.parent / "parity_report.json", {"examples": reports, "dataset_sha256": dataset["dataset_sha256"], "scope": "CPU validation only; Android untested"})
    display(reports)
    return exported.parent / "parity_report.json"
run_action("export_parity", RUN_EXPORT_PARITY, do_parity, "Produce a real export and reviewed validation data, then enable RUN_EXPORT_PARITY.")


In [ ]:
def package_handoff(destination, paths, metadata):
    """Explicit allowlist only: no automatic inclusion of raw photographs."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    artifacts = []
    for name, source in paths.items():
        source = Path(source)
        if not source.is_file():
            continue
        target = inside(destination, name)
        copy_once(source, target)
        artifacts.append({"path": name, "sha256": sha256(target), "bytes": target.stat().st_size})
    save_json(destination / "artifact_manifest.json", {**metadata, "artifacts": artifacts})
    return destination


In [ ]:
def do_handoff():
    if not NOTEBOOK_COPY.is_file(): raise ValueError(f"Save this notebook at {NOTEBOOK_COPY} before bundling")
    paths = {"Pluto_Propeller_Dataset_and_Training.ipynb": NOTEBOOK_COPY, "image_metadata.csv": METADATA_PATH, "seed_correspondence.csv": SEED_MAP_PATH,
        "split_manifest.json": SPLIT_PATH, "reference.json": REFERENCE_PATH, "class_confirmation.json": CLASS_CONFIRMATION_PATH, "user_body_convention.json": CONVENTION_PATH,
        "environment.json": REPORT_DIR / "environment.json", "environment-requirements.txt": REPORT_DIR / "environment-requirements.txt"}
    for i, manifest in enumerate(chosen_manifests()):
        paths[f"approved/{i}/manifest.json"] = Path(manifest)
        for row in load_json(manifest)["records"]: paths[f"approved/{i}/labels/{row['image_id']}.txt"] = Path(row["label_path"])
    for stage, override in (("bootstrap", BOOTSTRAP_RUN_MANIFEST), ("final", FINAL_RUN_MANIFEST)):
        try:
            manifest, run = resolve_run(PROJECT, stage, override)
            paths.update({f"{stage}/run_manifest.json": manifest, f"{stage}/best.pt": Path(run["best_pt"]), f"{stage}/last.pt": Path(run["last_pt"]),
                          f"{stage}/data.yaml": manifest.parent / "data.yaml", f"{stage}/results.csv": Path(run["best_pt"]).parent.parent / "results.csv"})
        except (ValueError, OSError): pass
    for split in ("val", "test"):
        pointer = PROJECT / "runs" / f"latest_evaluation_{split}.json"
        if pointer.is_file(): paths[f"evaluation/{split}.json"] = Path(load_json(pointer)["report_path"])
    pointer = PROJECT / "exports/latest_onnx.json"
    if pointer.is_file():
        exported = Path(load_json(pointer)["path"])
        paths.update({"best.onnx": exported, "export_manifest.json": exported.parent / "export_manifest.json", "parity_report.json": exported.parent / "parity_report.json"})
    readme = REPORT_DIR / "HANDOFF_README.txt"
    readme.write_text("Development detector handoff; no raw/reference photos included. Remap source/runtime paths explicitly. "
        "Notebook sections 8-10 explain labels/review/fixed TVT splits. Only actual artifacts/reports are included. "
        "Full verification, incorrect-installation sensitivity and Android execution are not established. "
        "Preprocessing/NMS/actual ONNX I/O are in section 17. Coordinate runtime/licences/resource access with app owner.\n", encoding="utf-8")
    paths["README.txt"] = readme
    output = package_handoff(PROJECT / "exports" / ("handoff_" + uuid.uuid4().hex), paths, {"created_at": utc_now(), "classes": CLASSES,
        "contract_version": "1.0.0", "private_photographs_included": False, "android_verified": False})
    print("Actual local handoff:", output)
    return output
run_action("handoff", RUN_HANDOFF_BUNDLE, do_handoff, "Save NOTEBOOK_COPY, then enable RUN_HANDOFF_BUNDLE for an allowlisted local bundle.")


## 18. Honest summary and remaining work

Run after any stage. Source files, distinct clean-source candidates, reference material, approved seeds, pending proposals, independently reviewed held-out pairs, actual execution this session, saved checkpoints and actual evaluation reports are separate fields. Hash-distinct files do not prove independent captures; missing sessions stay unverified. Reports/templates alone never imply training.

First run: install → settings → mount → sections 4–10. Inspect actual labeled/ contents and correspondence. Images-only references need real coordinates on matching clean originals through section 8, not training on drawn boxes/text. Confirm/review the existing ten, explicitly run A, review B, independently label C, then run D/evaluation when ready.

No synthetic photos or fake labels/weights/metrics are used. Physical class/front/motor/orientation/completeness evidence remains necessary. Correct supplied assembly data does not establish rejection of real incorrect assemblies. Colab Drive/UI, real photos, GPU training/resume, actual metrics, ONNX parity and Android require actual execution. Local fixtures test software only.

In [ ]:
try: summary_approved = current_approved()
except (ValueError, OSError) as exc:
    summary_approved = []
    print("Approved selection unavailable:", exc)
summary_rows, _ = refreshed()
clean = [r for r in summary_rows if r["role"] != "reference_only"]
refs = [r for r in summary_rows if r["role"] == "reference_only"]
seed_ids, seed_errors = selected_seed_ids(summary_rows, read_csv(SEED_MAP_PATH))
approved_ids = {r["image_id"] for r in summary_approved}
pending_ids = set()
for file in (PROJECT / "reviews/training_proposals").glob("*/image_review.csv"):
    pending_ids.update(r["image_id"] for r in read_csv(file) if r["image_id"] not in approved_ids)
saved_runs, evaluations = {}, {}
for stage, override in (("bootstrap", BOOTSTRAP_RUN_MANIFEST), ("final", FINAL_RUN_MANIFEST)):
    try:
        manifest, run = resolve_run(PROJECT, stage, override)
        saved_runs[stage] = {"manifest": str(manifest), "state": run["state"], "best_pt": run["best_pt"], "last_pt": run["last_pt"]}
    except (ValueError, OSError) as exc: saved_runs[stage] = {"state": "no_completed_checkpoint_available", "reason": str(exc)}
for split in ("val", "test"):
    pointer = PROJECT / "runs" / f"latest_evaluation_{split}.json"
    if pointer.is_file():
        report = Path(load_json(pointer)["report_path"])
        evaluations[split] = load_json(report) if report.is_file() else {"state": "recorded_report_missing"}
summary = {"source_images_found": len(summary_rows), "source_clean_candidates_by_split": dict(Counter(r["source_split"] for r in clean)),
    "unique_clean_candidates_exact_pixels": len({r["pixel_sha256"] for r in clean if not r["error"]}), "human_confirmed_clean_images_approved": len(summary_approved),
    "reference_overlays_or_material_found": len(refs), "approved_seed_labels": len(approved_ids & set(seed_ids)), "seed_correspondence_errors": seed_errors,
    "pending_training_proposal_images": len(pending_ids), "independently_reviewed_heldout_labels": dict(Counter(r["source_split"] for r in summary_approved if r["source_split"] in {"val", "test"} and r["review"].get("independent_review") == "true")),
    "execution_this_session": EXECUTION, "bootstrap_training_actually_ran": EXECUTION.get("bootstrap_training", {}).get("state") == "completed" or (RESUME_STAGE == "bootstrap" and EXECUTION.get("resume", {}).get("state") == "completed"),
    "final_training_actually_ran": EXECUTION.get("final_training", {}).get("state") == "completed" or (RESUME_STAGE == "final" and EXECUTION.get("resume", {}).get("state") == "completed"),
    "saved_completed_runs": saved_runs, "actual_evaluation_reports": evaluations, "session_independence": split_audit(summary_rows)["session_independence"], "reference_status": REFERENCE["status"],
    "limitations": ["distinct hashes do not prove capture independence", "burned-in references excluded from training", "real incorrect-installation performance unestablished", "front/motor/completeness/orientation verifier pending", "Android untested"]}
save_json(REPORT_DIR / "summary.json", summary)
print(json.dumps(summary, indent=2))
print("Persistent reports:", REPORT_DIR)
